# Bayan — Applied NLP Project

**Student:** Shouq Aldossari  
**Program:** SDA-AIE-211  
**Training Context:** SDAIA Academy

## Project Overview

This notebook contains the complete technical implementation of the Bayan project in one cumulative notebook.

The workflow is:

`Text → Preprocessing → Classification / NER / QA → Semantic Search → Evaluation → Optimisation → API`

The notebook follows the technical requirements **T1–T7** and includes the required tests, measurements, and final evidence.

### Runtime

Use the course reference environment: **Colab 2026.07 / Python 3.12.x**.

### Data

Data is loaded directly from the course repository. If a required file is missing, the notebook stops with an error.

The public `data/sample/` files are development / measured-smoke data. Final cohort target claims require the frozen assessment package when it is provided.

## 0. Environment Setup

These cells check the runtime, clone the course repository, install the pinned requirements, and verify package versions.

In [1]:
# Runtime compatibility check.
import sys, platform

print("Python:", sys.version)
print("Platform:", platform.platform())

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        "This project is aligned to the instructor-tested Colab 2026.07 runtime "
        "(Python 3.12.x). In Colab choose Runtime → Change runtime type → "
        "Runtime version → 2026.07, then reconnect and Run all."
    )

print("REFERENCE_RUNTIME_CHECK=PASS")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
REFERENCE_RUNTIME_CHECK=PASS


In [2]:
# Clone the course repository.
import subprocess
from pathlib import Path
import shutil, os

COURSE_REPO = "https://github.com/almiyead-rgb/bayan-applied-nlp-course.git"
COURSE_DIR = Path("/content/bayan-applied-nlp-course")

if COURSE_DIR.exists():
    shutil.rmtree(COURSE_DIR)

subprocess.check_call([
    "git", "clone", "--depth", "1",
    COURSE_REPO, str(COURSE_DIR)
])

required_paths = [
    COURSE_DIR / "requirements-day1.txt",
    COURSE_DIR / "requirements-day2.txt",
    COURSE_DIR / "requirements-day3.txt",
    COURSE_DIR / "requirements-day4.txt",
    COURSE_DIR / "data/sample/bayan_day2_classification.csv",
    COURSE_DIR / "data/sample/bayan_day2_ner.jsonl",
    COURSE_DIR / "data/sample/bayan_day2_qa.json",
    COURSE_DIR / "data/sample/bayan_day3_arabic.csv",
    COURSE_DIR / "data/sample/bayan_day3_cases.csv",
    COURSE_DIR / "data/sample/bayan_day3_queries.jsonl",
    COURSE_DIR / "data/sample/bayan_day3_predictions.csv",
    COURSE_DIR / "src/bayan/preprocessing.py",
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise RuntimeError(f"Required course files are missing: {missing}")

print("COURSE_REPOSITORY=PASS")

COURSE_REPOSITORY=PASS


In [3]:
# Install the pinned course versions.
import subprocess, sys

requirement_files = [
    COURSE_DIR / "requirements-day1.txt",
    COURSE_DIR / "requirements-day2.txt",
    COURSE_DIR / "requirements-day3.txt",
    COURSE_DIR / "requirements-day4.txt",
]

cmd = [sys.executable, "-m", "pip", "install", "--quiet"]
for req in requirement_files:
    cmd += ["-r", str(req)]

subprocess.check_call(cmd)

print("COURSE_REQUIREMENTS_INSTALL=PASS")

COURSE_REQUIREMENTS_INSTALL=PASS


In [4]:
# Import/version audit after installation.
import importlib.metadata as mdist

EXPECTED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "spacy": "3.8.7",
    "scikit-learn": "1.9.0",
    "camel-tools": "1.6.0",
    "sentence-transformers": "6.0.0",
    "faiss-cpu": "1.15.0",
    "onnx": "1.22.0",
    "onnxruntime": "1.29.0",
    "fastapi": "0.141.1",
    "psutil": "7.2.2",
}

actual = {}
for package, expected in EXPECTED.items():
    actual[package] = mdist.version(package)
    if actual[package] != expected:
        raise RuntimeError(
            f"{package}: expected {expected}, got {actual[package]}"
        )

print("PINNED_VERSIONS=PASS")
print(actual)

PINNED_VERSIONS=PASS
{'transformers': '5.15.1', 'tokenizers': '0.22.2', 'spacy': '3.8.7', 'scikit-learn': '1.9.0', 'camel-tools': '1.6.0', 'sentence-transformers': '6.0.0', 'faiss-cpu': '1.15.0', 'onnx': '1.22.0', 'onnxruntime': '1.29.0', 'fastapi': '0.141.1', 'psutil': '7.2.2'}


## 1. Imports and Reproducibility

Imports the project utilities, fixes the random seed, and selects CPU or GPU automatically.

In [5]:
import sys, os, json, math, time, random, hashlib, unicodedata, copy
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import re
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def reset_reproducibility(seed=SEED):
    """Reset Python/NumPy/PyTorch RNGs before each trainable model."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_reproducibility(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SRC_ROOT = COURSE_DIR / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bayan.preprocessing import (
    build_text_record,
    mask_pii,
    normalize_arabic,
    normalize_whitespace,
)
from bayan.tokenization import token_fertility, truncation_rate
from bayan.attention import scaled_dot_product_attention
from bayan.splits import validate_predefined_splits
from bayan.ner_alignment import align_word_labels, entity_f1
from bayan.qa_postprocess import best_span
from bayan.retrieval import (
    l2_normalize,
    retrieval_metrics,
    tune_no_answer_threshold,
)
from bayan.error_analysis import (
    summarise_taxonomy,
    behavioural_pass_rate,
)
from bayan.eval_stats import bootstrap_ci, sliced_metric_report
from bayan.benchmarking import (
    benchmark_callable,
    quality_tax,
    artifact_size_mb,
    assess_budget,
)
from bayan.serving import (
    sha256_file,
    infer_language,
    validate_request_text,
)

print("DEVICE =", DEVICE)
print("COURSE_MODULES=PASS")

DEVICE = cpu
COURSE_MODULES=PASS


# DAY 1 — Gate A: Ingest

## 2. T1 — Protected, Versioned Preprocessing

Bayan uses a two-copy contract:

- `display_text`: preserved for human review
- `model_text`: masked and normalised for model input

The preprocessing version is kept explicit so the same transformation can be used in training, evaluation, and serving.

In [6]:
PREPROCESSING_VERSION = "shouq-bayan-preprocess-v1.0"

def prepare_bayan_text(text: str, language: str):
    record = build_text_record(
        text,
        language=language,
        remove_diacritics=False,
        normalize_alef=False,
        normalize_ya=False,
    )
    return {
        "display_text": record.display_text,
        "model_text": record.model_text,
        "language": language,
        "preprocessing_version": PREPROCESSING_VERSION,
    }

sample = prepare_bayan_text(
    "راسل test@example.com أو اتصل 0500000000",
    "ar"
)
print(sample)
assert "test@example.com" in sample["display_text"]
assert "[EMAIL]" in sample["model_text"]
assert "[PHONE]" in sample["model_text"]
print("TWO_COPY_CONTRACT=PASS")

{'display_text': 'راسل test@example.com أو اتصل 0500000000', 'model_text': 'راسل [EMAIL] أو اتصل [PHONE]', 'language': 'ar', 'preprocessing_version': 'shouq-bayan-preprocess-v1.0'}
TWO_COPY_CONTRACT=PASS


### Golden Tests and PII Canaries

These canaries are part of the technical evidence for T1. The notebook stops if any declared PII pattern is not masked.

In [7]:
PII_CANARIES = [
    ("learner@example.com", "en", "[EMAIL]"),
    ("راسل learner@example.com", "ar", "[EMAIL]"),
    ("0500000000", "ar", "[PHONE]"),
    ("+966500000000", "ar", "[PHONE]"),
]

canary_pass = []
for text, language, marker in PII_CANARIES:
    model_text = prepare_bayan_text(text, language)["model_text"]
    passed = marker in model_text
    canary_pass.append(passed)
    print(marker, "PASS" if passed else "FAIL", model_text)

assert all(canary_pass)
print("PII_MASKING_RECALL=1.0")

[EMAIL] PASS [EMAIL]
[EMAIL] PASS راسل [EMAIL]
[PHONE] PASS [PHONE]
[PHONE] PASS [PHONE]
PII_MASKING_RECALL=1.0


In [8]:
# Train/serve skew canaries.
SKEW_CASES = [
    ("الخـدمة   واضحة", "ar"),
    ("The   service is clear", "en"),
    ("راسل test@example.com", "ar"),
]

for text, language in SKEW_CASES:
    training_view = prepare_bayan_text(text, language)["model_text"]
    serving_view = prepare_bayan_text(text, language)["model_text"]
    assert training_view == serving_view

print("TRAIN_SERVE_SKEW=PASS")

TRAIN_SERVE_SKEW=PASS


## 3. T1 — Tokenisation and Length Evidence

Measures token fertility, token lengths, truncation rate, and selects `max_length` from the data.

In [9]:
from transformers import AutoTokenizer, AutoModel

ENCODER_MODEL_ID = "distilbert/distilbert-base-multilingual-cased"

encoder_tokenizer = AutoTokenizer.from_pretrained(
    ENCODER_MODEL_ID,
    use_fast=True,
)
encoder_model = AutoModel.from_pretrained(
    ENCODER_MODEL_ID
).to(DEVICE)
encoder_model.eval()

day1_df = pd.read_csv(
    COURSE_DIR / "data/sample/bayan_day1_sample.csv"
)
classification_df = pd.read_csv(
    COURSE_DIR / "data/sample/bayan_day2_classification.csv"
)

tokenise = lambda text: encoder_tokenizer.tokenize(text)

fertility_rows = []
for language, part in classification_df.groupby("language"):
    values = [
        token_fertility(text, tokenise)
        for text in part["text"]
    ]
    fertility_rows.append({
        "language": language,
        "n": len(values),
        "mean_fertility": float(np.mean(values)),
        "max_fertility": float(np.max(values)),
    })

fertility_report = pd.DataFrame(fertility_rows)
display(fertility_report)

lengths = np.array([
    len(
        encoder_tokenizer(
            text,
            add_special_tokens=True,
        )["input_ids"]
    )
    for text in classification_df["text"]
])

MAX_LENGTH = int(
    min(
        128,
        max(32, math.ceil(np.percentile(lengths, 95)))
    )
)

observed_truncation_rate = float(
    np.mean(lengths > MAX_LENGTH)
)

print({
    "p50": float(np.percentile(lengths, 50)),
    "p95": float(np.percentile(lengths, 95)),
    "max": int(lengths.max()),
    "chosen_max_length": MAX_LENGTH,
    "truncation_rate": observed_truncation_rate,
})

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,language,n,mean_fertility,max_fertility
0,ar,20,2.428333,3.25
1,en,20,1.106310,1.40


{'p50': 11.0, 'p95': 16.0, 'max': 18, 'chosen_max_length': 32, 'truncation_rate': 0.0}


## 4. T2 — Attention and Transformer Checks

Checks Q/K/V dimensions, masks, attention weights, multi-head shapes, encoder flow, and interpretation limits.

In [10]:
rng = np.random.default_rng(SEED)

q = np.array([[1.0, 0.0], [0.0, 1.0]])
k = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
v = np.array([[10.0, 0.0], [0.0, 10.0], [5.0, 5.0]])

attention_output, attention_weights = scaled_dot_product_attention(
    q, k, v
)

assert attention_output.shape == (2, 2)
assert np.allclose(
    attention_weights.sum(axis=-1),
    1.0
)

keep_mask = np.tril(
    np.ones((2, 3), dtype=bool)
)
masked_output, masked_weights = scaled_dot_product_attention(
    q, k, v,
    keep_mask=keep_mask,
)

assert masked_weights[0,1] == 0.0
assert masked_weights[0,2] == 0.0

print("ATTENTION_SHAPES=PASS")
print("MASK_BEFORE_SOFTMAX=PASS")

ATTENTION_SHAPES=PASS
MASK_BEFORE_SOFTMAX=PASS


In [11]:
# Explicit Q/K/V and multi-head shape journey.
BATCH, TOKENS, D_MODEL, HEADS = 2, 5, 12, 3
D_HEAD = D_MODEL // HEADS

x = rng.normal(size=(BATCH, TOKENS, D_MODEL))

def split_heads(x, num_heads):
    b, n, d = x.shape
    if d % num_heads:
        raise ValueError("d_model must be divisible by num_heads")
    d_head = d // num_heads
    return x.reshape(b, n, num_heads, d_head).transpose(0,2,1,3)

def combine_heads(x):
    b, h, n, d = x.shape
    return x.transpose(0,2,1,3).reshape(b,n,h*d)

heads = split_heads(x, HEADS)
round_trip = combine_heads(heads)

assert heads.shape == (BATCH, HEADS, TOKENS, D_HEAD)
assert np.allclose(round_trip, x)

print("Q/K/V conceptual shape = (batch, heads, tokens, d_head)")
print("Multi-head shape:", heads.shape)
print("MULTI_HEAD_SHAPES=PASS")

Q/K/V conceptual shape = (batch, heads, tokens, d_head)
Multi-head shape: (2, 3, 5, 4)
MULTI_HEAD_SHAPES=PASS


### Encoder Flow and Interpretation Limit

A Transformer encoder maps token IDs to embeddings, adds position information, applies repeated self-attention and feed-forward blocks with residual connections and layer normalisation, and returns contextual token representations.

**Important limitation:** attention weights are internal routing weights. They are not guaranteed explanations of model decisions, and an individual attention head should not be labelled as a linguistic feature without separate evidence.

In [12]:
encoded = encoder_tokenizer(
    ["الخدمة لم تتأخر.", "The service was not delayed."],
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    hidden = encoder_model(**encoded).last_hidden_state

assert hidden.shape[:2] == encoded["input_ids"].shape
print("ENCODER_OUTPUT_SHAPE =", tuple(hidden.shape))
print("ENCODER_FLOW=PASS")

ENCODER_OUTPUT_SHAPE = (2, 10, 768)
ENCODER_FLOW=PASS


# DAY 2 — Gate B: Tasks

## 5. T3 — Official Grouped Split

The course file already contains `train`, `validation`, and `test` plus `group_id`. The group-isolation contract is validated before any model is trained.

In [13]:
classification_rows = classification_df.to_dict("records")

split_report = validate_predefined_splits(
    classification_rows,
    group_key="group_id",
    split_key="split",
    label_key="topic",
    require_all_labels=True,
)

print(split_report)
assert split_report["group_overlap"] == 0

train_rows = [
    row for row in classification_rows
    if row["split"] == "train"
]
validation_rows = [
    row for row in classification_rows
    if row["split"] == "validation"
]
test_rows = [
    row for row in classification_rows
    if row["split"] == "test"
]

print("GROUPED_SPLIT=PASS")

{'rows_per_split': {'train': 24, 'validation': 8, 'test': 8}, 'groups': 20, 'labels': ['digital_service', 'health', 'permit', 'transport'], 'group_overlap': 0}
GROUPED_SPLIT=PASS


## 6. T3 — Topic and Sentiment Baselines

Builds separate TF-IDF + LinearSVC baselines for topic and sentiment and reports Macro-F1.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

def fit_tfidf_baseline(target):
    labels = sorted(
        classification_df[target].unique()
    )
    model = make_pipeline(
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3,5),
            min_df=1,
        ),
        LinearSVC(random_state=SEED),
    )
    model.fit(
        [r["text"] for r in train_rows],
        [r[target] for r in train_rows],
    )
    pred = model.predict(
        [r["text"] for r in validation_rows]
    )
    score = f1_score(
        [r[target] for r in validation_rows],
        pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )
    return model, labels, score

topic_baseline, TOPIC_LABELS, topic_baseline_f1 = fit_tfidf_baseline("topic")
sentiment_baseline, SENTIMENT_LABELS, sentiment_baseline_f1 = fit_tfidf_baseline("sentiment")

print({
    "topic_baseline_val_macro_f1": topic_baseline_f1,
    "sentiment_baseline_val_macro_f1": sentiment_baseline_f1,
})

{'topic_baseline_val_macro_f1': 0.6666666666666666, 'sentiment_baseline_val_macro_f1': 0.07407407407407407}


## 7. T3 — Topic and Sentiment Transformer Models

Fine-tunes two independent Transformer heads: one for topic and one for sentiment.

In [15]:
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification

CLASSIFIER_MODEL_ID = "distilbert/distilbert-base-multilingual-cased"
CLASSIFIER_TOKENIZER = AutoTokenizer.from_pretrained(
    CLASSIFIER_MODEL_ID,
    use_fast=True,
)

# Source-aligned smoke workload:
# 24 train rows / batch 4 = 6 optimiser steps per epoch.
# 12 epochs = 72 steps, matching the verified course run shape.
TRAINING_MODE = "partial_finetune_reference"
BATCH_SIZE = 4
NUM_EPOCHS = 12
CLASSIFIER_LR = 1e-4

def iter_batches(examples, batch_size, *, shuffle=False, seed=SEED):
    indexes = np.arange(len(examples))
    if shuffle:
        np.random.default_rng(seed).shuffle(indexes)
    for start in range(0, len(indexes), batch_size):
        yield [examples[i] for i in indexes[start:start + batch_size]]

def train_sequence_head(target, seed=SEED):
    reset_reproducibility(seed)

    labels = sorted(classification_df[target].unique())
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER_MODEL_ID,
        num_labels=len(labels),
        label2id=label2id,
        id2label=id2label,
    )

    # Keep the pretrained encoder stable; train last block + task head.
    for p in model.base_model.parameters():
        p.requires_grad = False
    for p in model.base_model.transformer.layer[-1].parameters():
        p.requires_grad = True

    model.to(DEVICE)

    def encode_batch(batch):
        encoded = CLASSIFIER_TOKENIZER(
            [row["text"] for row in batch],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded["labels"] = torch.tensor(
            [label2id[row[target]] for row in batch],
            dtype=torch.long,
        )
        return {k: v.to(DEVICE) for k, v in encoded.items()}

    def predict(examples):
        model.eval()
        output_ids = []
        with torch.no_grad():
            for batch in iter_batches(examples, BATCH_SIZE):
                encoded = encode_batch(batch)
                logits = model(**encoded).logits
                output_ids.extend(logits.argmax(-1).cpu().tolist())
        return [id2label[i] for i in output_ids]

    baseline_model, _, baseline_val_f1 = fit_tfidf_baseline(target)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=CLASSIFIER_LR)

    best_val_f1 = -1.0
    best_epoch = 0
    best_state = None
    history = []
    train_steps = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_losses = []

        for batch in iter_batches(
            train_rows,
            BATCH_SIZE,
            shuffle=True,
            seed=seed + epoch + 1,
        ):
            optimizer.zero_grad(set_to_none=True)
            encoded = encode_batch(batch)
            out = model(**encoded)

            if not torch.isfinite(out.loss):
                raise RuntimeError(f"Non-finite {target} training loss")

            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()

            epoch_losses.append(float(out.loss.detach().cpu()))
            train_steps += 1

        val_pred = predict(validation_rows)
        val_f1 = f1_score(
            [r[target] for r in validation_rows],
            val_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        )

        history.append({
            "epoch": epoch + 1,
            "mean_loss": float(np.mean(epoch_losses)),
            "validation_macro_f1": float(val_f1),
        })

        if val_f1 > best_val_f1:
            best_val_f1 = float(val_f1)
            best_epoch = epoch + 1
            best_state = {
                name: p.detach().cpu().clone()
                for name, p in model.named_parameters()
                if p.requires_grad
            }

        print(
            f"{target} epoch={epoch + 1:02d} "
            f"loss={np.mean(epoch_losses):.4f} "
            f"val_macro_f1={val_f1:.4f}"
        )

    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in best_state:
                p.copy_(best_state[name].to(p.device))

    selected_val_pred = predict(validation_rows)
    selected_val_f1 = f1_score(
        [r[target] for r in validation_rows],
        selected_val_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

    return {
        "model": model,
        "predict": predict,
        "labels": labels,
        "label2id": label2id,
        "id2label": id2label,
        "baseline": baseline_model,
        "baseline_val_f1": float(baseline_val_f1),
        "transformer_val_f1": float(selected_val_f1),
        "history": history,
        "selected_epoch": int(best_epoch),
        "train_steps": int(train_steps),
    }

TOPIC_ARTIFACT = train_sequence_head("topic", seed=SEED)
SENTIMENT_ARTIFACT = train_sequence_head("sentiment", seed=SEED)

# Validation-only deployment decision for sentiment.
SENTIMENT_SERVING_BACKEND = (
    "transformer"
    if SENTIMENT_ARTIFACT["transformer_val_f1"]
       >= SENTIMENT_ARTIFACT["baseline_val_f1"]
    else "tfidf_baseline"
)

print({
    "training_mode": TRAINING_MODE,
    "epochs": NUM_EPOCHS,
    "topic": {
        "selected_epoch": TOPIC_ARTIFACT["selected_epoch"],
        "train_steps": TOPIC_ARTIFACT["train_steps"],
        "baseline_val_f1": TOPIC_ARTIFACT["baseline_val_f1"],
        "transformer_val_f1": TOPIC_ARTIFACT["transformer_val_f1"],
        "delta": (
            TOPIC_ARTIFACT["transformer_val_f1"]
            - TOPIC_ARTIFACT["baseline_val_f1"]
        ),
    },
    "sentiment": {
        "selected_epoch": SENTIMENT_ARTIFACT["selected_epoch"],
        "train_steps": SENTIMENT_ARTIFACT["train_steps"],
        "baseline_val_f1": SENTIMENT_ARTIFACT["baseline_val_f1"],
        "transformer_val_f1": SENTIMENT_ARTIFACT["transformer_val_f1"],
        "delta": (
            SENTIMENT_ARTIFACT["transformer_val_f1"]
            - SENTIMENT_ARTIFACT["baseline_val_f1"]
        ),
        "serving_backend": SENTIMENT_SERVING_BACKEND,
    },
})

assert TOPIC_ARTIFACT["train_steps"] == 72
assert SENTIMENT_ARTIFACT["train_steps"] == 72
print("CLASSIFICATION_REFERENCE_WORKLOAD=PASS")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


topic epoch=01 loss=1.4031 val_macro_f1=0.2917
topic epoch=02 loss=1.3156 val_macro_f1=0.4167
topic epoch=03 loss=1.2396 val_macro_f1=0.4167
topic epoch=04 loss=1.1272 val_macro_f1=0.6250
topic epoch=05 loss=0.9523 val_macro_f1=0.6250
topic epoch=06 loss=0.7202 val_macro_f1=0.7500
topic epoch=07 loss=0.4770 val_macro_f1=0.8667
topic epoch=08 loss=0.2488 val_macro_f1=0.8667
topic epoch=09 loss=0.1160 val_macro_f1=1.0000
topic epoch=10 loss=0.0552 val_macro_f1=1.0000
topic epoch=11 loss=0.0225 val_macro_f1=1.0000
topic epoch=12 loss=0.0115 val_macro_f1=0.8667


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


sentiment epoch=01 loss=1.1007 val_macro_f1=0.1333
sentiment epoch=02 loss=1.0401 val_macro_f1=0.1333
sentiment epoch=03 loss=0.9709 val_macro_f1=0.1333
sentiment epoch=04 loss=0.8677 val_macro_f1=0.2434
sentiment epoch=05 loss=0.7335 val_macro_f1=0.3333
sentiment epoch=06 loss=0.5441 val_macro_f1=0.3333
sentiment epoch=07 loss=0.2889 val_macro_f1=0.4127
sentiment epoch=08 loss=0.1356 val_macro_f1=0.4127
sentiment epoch=09 loss=0.0563 val_macro_f1=0.3333
sentiment epoch=10 loss=0.0136 val_macro_f1=0.4127
sentiment epoch=11 loss=0.0088 val_macro_f1=0.4127
sentiment epoch=12 loss=0.0040 val_macro_f1=0.4127
{'training_mode': 'partial_finetune_reference', 'epochs': 12, 'topic': {'selected_epoch': 9, 'train_steps': 72, 'baseline_val_f1': 0.6666666666666666, 'transformer_val_f1': 1.0, 'delta': 0.33333333333333337}, 'sentiment': {'selected_epoch': 7, 'train_steps': 72, 'baseline_val_f1': 0.07407407407407407, 'transformer_val_f1': 0.41269841269841273, 'delta': 0.33862433862433866, 'serving_bac

### Final Classification Smoke Test — Open Once

Model choices are fixed using the validation split first.  
This cell then opens the small public test split once and records the final `MEASURED_SMOKE` evidence for topic and sentiment. The result is not used to tune the models.

In [16]:
FROZEN_TEST_OPENED = True

def final_classification_test(target, artifact):
    truth = [r[target] for r in test_rows]
    baseline_pred = artifact["baseline"].predict(
        [r["text"] for r in test_rows]
    ).tolist()
    transformer_pred = artifact["predict"](test_rows)

    labels = artifact["labels"]

    return {
        "target": target,
        "n": len(test_rows),
        "baseline_macro_f1": float(
            f1_score(
                truth,
                baseline_pred,
                labels=labels,
                average="macro",
                zero_division=0,
            )
        ),
        "transformer_macro_f1": float(
            f1_score(
                truth,
                transformer_pred,
                labels=labels,
                average="macro",
                zero_division=0,
            )
        ),
        "transformer_accuracy": float(
            accuracy_score(truth, transformer_pred)
        ),
        "truth": truth,
        "baseline_prediction": baseline_pred,
        "transformer_prediction": transformer_pred,
    }

TOPIC_TEST = final_classification_test(
    "topic",
    TOPIC_ARTIFACT,
)
SENTIMENT_TEST = final_classification_test(
    "sentiment",
    SENTIMENT_ARTIFACT,
)

PROJECT_TEST_PREDICTIONS = pd.DataFrame([
    {
        "example_id": row["example_id"],
        "language": row["language"],
        "text": row["text"],
        "topic_truth": row["topic"],
        "topic_prediction": TOPIC_TEST["transformer_prediction"][i],
        "sentiment_truth": row["sentiment"],
        "sentiment_prediction": SENTIMENT_TEST["transformer_prediction"][i],
    }
    for i, row in enumerate(test_rows)
])

print("TOPIC_TEST_MEASURED_SMOKE =", TOPIC_TEST)
print("SENTIMENT_TEST_MEASURED_SMOKE =", SENTIMENT_TEST)
display(PROJECT_TEST_PREDICTIONS)

print("FROZEN_TEST_FINAL_SMOKE=PASS")

TOPIC_TEST_MEASURED_SMOKE = {'target': 'topic', 'n': 8, 'baseline_macro_f1': 0.7333333333333333, 'transformer_macro_f1': 0.8666666666666667, 'transformer_accuracy': 0.875, 'truth': ['digital_service', 'digital_service', 'permit', 'permit', 'health', 'health', 'transport', 'transport'], 'baseline_prediction': ['digital_service', 'health', 'permit', 'permit', 'health', 'health', 'transport', 'permit'], 'transformer_prediction': ['health', 'digital_service', 'permit', 'permit', 'health', 'health', 'transport', 'transport']}
SENTIMENT_TEST_MEASURED_SMOKE = {'target': 'sentiment', 'n': 8, 'baseline_macro_f1': 0.6666666666666666, 'transformer_macro_f1': 0.5333333333333333, 'transformer_accuracy': 0.75, 'truth': ['positive', 'positive', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative'], 'baseline_prediction': ['positive', 'positive', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative'], 'transformer_prediction': ['positive', 'positive', 'negative', 'ne

,example_id,language,text,topic_truth,topic_prediction,sentiment_truth,sentiment_prediction
0,D-009,ar,الخدمة الإلكترونية واضحة وسريعة,digital_service,health,positive,positive
1,D-010,en,The online service is clear and fast,digital_service,digital_service,positive,positive
2,D-019,ar,لم أجد رقم طلب التصريح,permit,permit,negative,negative
3,D-020,en,I could not find the permit request number,permit,permit,negative,neutral
4,D-029,ar,تم إلغاء موعد العيادة دون إشعار,health,health,negative,positive
5,D-030,en,The clinic appointment was cancelled without n...,health,health,negative,negative
6,D-039,ar,توقفت الحافلة قبل المحطة المطلوبة,transport,transport,negative,negative
7,D-040,en,The bus stopped before the requested station,transport,transport,negative,negative


FROZEN_TEST_FINAL_SMOKE=PASS


### Save the Project Classifier Artefact

The topic classifier is saved locally and reused in the Day 4 optimisation section. This makes T6 benchmark the **actual project classifier artefact**, not a separate systems-smoke model.

In [17]:
PROJECT_ARTIFACT_DIR = Path("/content/bayan_project_artifact")
PROJECT_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TOPIC_ARTIFACT["model"].save_pretrained(PROJECT_ARTIFACT_DIR)
CLASSIFIER_TOKENIZER.save_pretrained(PROJECT_ARTIFACT_DIR)

validation_csv = Path("/content/bayan_project_validation.csv")
pd.DataFrame([
    {
        "example_id": r["example_id"],
        "split": "validation",
        "language": r["language"],
        "text": r["text"],
        "label": r["topic"],
    }
    for r in validation_rows
]).to_csv(validation_csv, index=False)

print("PROJECT_MODEL_SOURCE =", PROJECT_ARTIFACT_DIR)
print("PROJECT_VALIDATION_CSV =", validation_csv)
print("PROJECT_ARTIFACT_HANDOFF=PASS")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

PROJECT_MODEL_SOURCE = /content/bayan_project_artifact
PROJECT_VALIDATION_CSV = /content/bayan_project_validation.csv
PROJECT_ARTIFACT_HANDOFF=PASS


## 8. T3 — Named Entity Recognition

Aligns BIO labels to subwords, trains the NER model, and calculates entity-level precision, recall, and F1.

In [18]:
ner_path = COURSE_DIR / "data/sample/bayan_day2_ner.jsonl"
with ner_path.open(encoding="utf-8") as f:
    NER_ROWS = [json.loads(line) for line in f if line.strip()]

NER_LABELS = [
    "O",
    "B-SERVICE","I-SERVICE",
    "B-LOCATION","I-LOCATION",
    "B-DATE","I-DATE",
    "B-REF_NUM","I-REF_NUM",
    "B-ORG","I-ORG",
]
NER_LABEL2ID = {label:i for i,label in enumerate(NER_LABELS)}
NER_ID2LABEL = {i:label for label,i in NER_LABEL2ID.items()}

for row in NER_ROWS:
    assert len(row["tokens"]) == len(row["ner_tags"])
    assert set(row["ner_tags"]) <= set(NER_LABELS)

print("NER_DATA_CONTRACT=PASS", len(NER_ROWS))

NER_DATA_CONTRACT=PASS 12


In [19]:
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)

NER_TOKENIZER = AutoTokenizer.from_pretrained(
    CLASSIFIER_MODEL_ID,
    use_fast=True,
)

reset_reproducibility(SEED)

NER_MODEL = AutoModelForTokenClassification.from_pretrained(
    CLASSIFIER_MODEL_ID,
    num_labels=len(NER_LABELS),
    label2id=NER_LABEL2ID,
    id2label=NER_ID2LABEL,
)

for p in NER_MODEL.base_model.parameters():
    p.requires_grad = False
for p in NER_MODEL.base_model.transformer.layer[-1].parameters():
    p.requires_grad = True

NER_MODEL.to(DEVICE)

def encode_ner(row):
    encoded = NER_TOKENIZER(
        row["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=64,
    )
    word_labels = [
        NER_LABEL2ID[tag]
        for tag in row["ner_tags"]
    ]
    encoded["labels"] = align_word_labels(
        encoded.word_ids(),
        word_labels,
    )
    return dict(encoded)

ner_train_rows = [r for r in NER_ROWS if r["split"] == "train"]
ner_validation_rows = [r for r in NER_ROWS if r["split"] == "validation"]
ner_test_rows = [r for r in NER_ROWS if r["split"] == "test"]

ner_train_features = [encode_ner(r) for r in ner_train_rows]
ner_collator = DataCollatorForTokenClassification(
    tokenizer=NER_TOKENIZER
)

# 8 train examples / batch 2 = 4 steps per epoch; 12 epochs = 48 steps.
NER_EPOCHS = 12
NER_LR = 1e-4

ner_optimizer = AdamW(
    [p for p in NER_MODEL.parameters() if p.requires_grad],
    lr=NER_LR,
)

ner_losses = []
NER_STEPS = 0

for epoch in range(NER_EPOCHS):
    loader = DataLoader(
        ner_train_features,
        batch_size=2,
        shuffle=True,
        collate_fn=ner_collator,
        generator=torch.Generator().manual_seed(SEED + epoch + 1),
    )

    NER_MODEL.train()
    epoch_losses = []

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        ner_optimizer.zero_grad(set_to_none=True)

        out = NER_MODEL(**batch)
        if not torch.isfinite(out.loss):
            raise RuntimeError("Non-finite NER loss")

        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in NER_MODEL.parameters() if p.requires_grad],
            1.0,
        )
        ner_optimizer.step()

        value = float(out.loss.detach().cpu())
        epoch_losses.append(value)
        ner_losses.append(value)
        NER_STEPS += 1

    print(
        f"NER epoch={epoch + 1:02d} "
        f"loss={np.mean(epoch_losses):.4f}"
    )

assert NER_STEPS == 48
print("NER_TRAINING=PASS", {
    "epochs": NER_EPOCHS,
    "steps": NER_STEPS,
    "mode": "partial_finetune_reference",
})

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NER epoch=01 loss=2.2637
NER epoch=02 loss=1.8038
NER epoch=03 loss=1.5641
NER epoch=04 loss=1.3335
NER epoch=05 loss=1.0610
NER epoch=06 loss=0.7984
NER epoch=07 loss=0.6044
NER epoch=08 loss=0.4012
NER epoch=09 loss=0.2601
NER epoch=10 loss=0.1418
NER epoch=11 loss=0.0711
NER epoch=12 loss=0.0427
NER_TRAINING=PASS {'epochs': 12, 'steps': 48, 'mode': 'partial_finetune_reference'}


In [20]:
def _bio_parts(label):
    if label == "O":
        return "O", None
    prefix, entity_type = label.split("-", 1)
    return prefix, entity_type

def _valid_bio_transition(previous_label, current_label):
    current_prefix, current_type = _bio_parts(current_label)

    if current_prefix in {"O", "B"}:
        return True

    # I-X is legal only after B-X or I-X.
    previous_prefix, previous_type = _bio_parts(previous_label)
    return (
        current_prefix == "I"
        and previous_prefix in {"B", "I"}
        and previous_type == current_type
    )

def bio_viterbi_decode(word_logits):
    """
    Decode one word-level logit matrix while enforcing legal BIO transitions.
    The decoder uses model scores only; no example-specific rules are used.
    """
    scores = torch.log_softmax(
        torch.as_tensor(word_logits, dtype=torch.float32),
        dim=-1,
    ).cpu().numpy()

    n_words, n_labels = scores.shape
    neg_inf = -1e30

    dp = np.full((n_words, n_labels), neg_inf, dtype=np.float64)
    back = np.full((n_words, n_labels), -1, dtype=np.int64)

    # First token cannot start with I-*.
    for current_id in range(n_labels):
        current_label = NER_ID2LABEL[current_id]
        prefix, _ = _bio_parts(current_label)
        if prefix != "I":
            dp[0, current_id] = scores[0, current_id]

    for position in range(1, n_words):
        for current_id in range(n_labels):
            current_label = NER_ID2LABEL[current_id]

            for previous_id in range(n_labels):
                previous_label = NER_ID2LABEL[previous_id]

                if not _valid_bio_transition(
                    previous_label,
                    current_label,
                ):
                    continue

                candidate = (
                    dp[position - 1, previous_id]
                    + scores[position, current_id]
                )

                if candidate > dp[position, current_id]:
                    dp[position, current_id] = candidate
                    back[position, current_id] = previous_id

    last_id = int(np.argmax(dp[-1]))
    path = [last_id]

    for position in range(n_words - 1, 0, -1):
        last_id = int(back[position, last_id])
        if last_id < 0:
            raise RuntimeError("BIO decoder backtracking failed.")
        path.append(last_id)

    path.reverse()
    return path

def ner_word_logits_from_tokens(tokens):
    encoded = NER_TOKENIZER(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    )

    word_ids = encoded.word_ids(0)
    inputs = {
        k: v.to(DEVICE)
        for k, v in encoded.items()
    }

    NER_MODEL.eval()
    with torch.no_grad():
        logits = NER_MODEL(**inputs).logits[0].cpu()

    first_subword_positions = []
    seen = set()

    for token_index, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen:
            continue
        seen.add(word_id)
        first_subword_positions.append(token_index)

    if len(first_subword_positions) != len(tokens):
        raise RuntimeError(
            "NER token/word alignment changed unexpectedly."
        )

    return logits[first_subword_positions].numpy()

def predict_ner(rows, *, constrained=True):
    truth_sequences = []
    pred_sequences = []

    for row in rows:
        tokens = row["tokens"]
        word_logits = ner_word_logits_from_tokens(tokens)

        if constrained:
            pred_ids = bio_viterbi_decode(word_logits)
        else:
            pred_ids = word_logits.argmax(axis=-1).tolist()

        truth_sequences.append(list(row["ner_tags"]))
        pred_sequences.append([
            NER_ID2LABEL[int(pred_id)]
            for pred_id in pred_ids
        ])

    return truth_sequences, pred_sequences

print("BIO_CONSTRAINED_DECODER=PASS")

BIO_CONSTRAINED_DECODER=PASS


In [21]:
# Compare raw greedy decoding with BIO-constrained decoding on validation.
ner_val_truth_greedy, ner_val_pred_greedy = predict_ner(
    ner_validation_rows,
    constrained=False,
)
NER_VALIDATION_GREEDY = entity_f1(
    ner_val_truth_greedy,
    ner_val_pred_greedy,
)

ner_val_truth, ner_val_pred = predict_ner(
    ner_validation_rows,
    constrained=True,
)
NER_VALIDATION_METRICS = entity_f1(
    ner_val_truth,
    ner_val_pred,
)

# The public test split is opened once after the decoder is fixed.
ner_test_truth, ner_test_pred = predict_ner(
    ner_test_rows,
    constrained=True,
)
NER_METRICS = entity_f1(
    ner_test_truth,
    ner_test_pred,
)

print("NER_VALIDATION_GREEDY =", NER_VALIDATION_GREEDY)
print(
    "NER_VALIDATION_BIO_CONSTRAINED =",
    NER_VALIDATION_METRICS,
)
print(
    "NER_TEST_ENTITY_METRICS_MEASURED_SMOKE =",
    NER_METRICS,
)

assert 0.0 <= NER_VALIDATION_METRICS["f1"] <= 1.0
assert 0.0 <= NER_METRICS["f1"] <= 1.0

print("NER_ENTITY_EVALUATION=PASS")

NER_VALIDATION_GREEDY = {'precision': 1.0, 'recall': 0.75, 'f1': 0.8571428571428571, 'true_entities': 4, 'predicted_entities': 3}
NER_VALIDATION_BIO_CONSTRAINED = {'precision': 1.0, 'recall': 0.75, 'f1': 0.8571428571428571, 'true_entities': 4, 'predicted_entities': 3}
NER_TEST_ENTITY_METRICS_MEASURED_SMOKE = {'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'true_entities': 4, 'predicted_entities': 2}
NER_ENTITY_EVALUATION=PASS


## 9. T3 — Extractive Question Answering

Trains extractive QA with answer spans and no-answer examples, then runs boundary tests.

In [22]:
qa_payload = json.loads(
    (COURSE_DIR / "data/sample/bayan_day2_qa.json").read_text(
        encoding="utf-8"
    )
)
QA_ROWS = qa_payload["examples"]

for row in QA_ROWS:
    if row["answer_text"] is not None:
        start = int(row["answer_start"])
        actual = row["context"][
            start:start+len(row["answer_text"])
        ]
        assert actual == row["answer_text"]

qa_train_rows = [r for r in QA_ROWS if r["split"]=="train"]
qa_validation_rows = [r for r in QA_ROWS if r["split"]=="validation"]
qa_test_rows = [r for r in QA_ROWS if r["split"]=="test"]

print({
    "qa_rows": len(QA_ROWS),
    "no_answer_examples": sum(r["answer_text"] is None for r in QA_ROWS),
})

{'qa_rows': 10, 'no_answer_examples': 2}


In [23]:
from transformers import AutoModelForQuestionAnswering

QA_TOKENIZER = CLASSIFIER_TOKENIZER
reset_reproducibility(SEED)

def build_cross_context_no_answer(rows, max_per_language=1):
    output=[]
    by_language={}
    for row in rows:
        by_language.setdefault(row["language"], []).append(row)
    for language, group in sorted(by_language.items()):
        produced=0
        ordered=sorted(group, key=lambda r:r["id"])
        for row in ordered:
            candidates=[c for c in ordered if c["id"]!=row["id"] and row["answer_text"] is not None and row["answer_text"] not in c["context"]]
            if not candidates: continue
            chosen=candidates[0]
            output.append({"split":row["split"],"language":language,"context":chosen["context"],"question":row["question"],"answer_text":None,"answer_start":None,"id":f"{row['id']}__NEG__{chosen['id']}","derived_no_answer":True})
            produced+=1
            if produced>=max_per_language: break
    return output

QA_TRAIN_NEGATIVES=build_cross_context_no_answer(qa_train_rows,1)
QA_TRAIN_AUGMENTED=qa_train_rows+QA_TRAIN_NEGATIVES

def build_validation_no_answer(validation_rows, train_rows):
    output=[]
    for vr in validation_rows:
        candidates=[r for r in train_rows if r["language"]==vr["language"] and r["answer_text"] is not None and r["answer_text"] not in vr["context"]]
        if not candidates: continue
        chosen=sorted(candidates,key=lambda r:r["id"])[0]
        output.append({"split":"validation","language":vr["language"],"context":vr["context"],"question":chosen["question"],"answer_text":None,"answer_start":None,"id":f"{vr['id']}__VAL_NEG__{chosen['id']}","derived_no_answer":True})
    return output

QA_VALIDATION_NEGATIVES=build_validation_no_answer(qa_validation_rows,qa_train_rows)
print("QA_AUGMENTATION =",{"official_train_answerable":len(qa_train_rows),"derived_train_no_answer":len(QA_TRAIN_NEGATIVES),"official_validation_answerable":len(qa_validation_rows),"derived_validation_no_answer":len(QA_VALIDATION_NEGATIVES),"test_used_for_training_or_tuning":False})

QA_MODEL=AutoModelForQuestionAnswering.from_pretrained(CLASSIFIER_MODEL_ID)
if DEVICE.type=="cuda":
    QA_TRAINING_MODE="full_finetune_gpu"; QA_LR=2e-5
else:
    QA_TRAINING_MODE="partial_finetune_cpu"; QA_LR=1e-4
    for p in QA_MODEL.base_model.parameters(): p.requires_grad=False
    for p in QA_MODEL.base_model.transformer.layer[-1].parameters(): p.requires_grad=True
QA_MODEL.to(DEVICE)
QA_MAX_LENGTH=96

def prepare_qa_batch(examples):
    enc=QA_TOKENIZER([r["question"].strip() for r in examples],[r["context"] for r in examples],padding=True,truncation="only_second",max_length=QA_MAX_LENGTH,return_offsets_mapping=True,return_tensors="pt")
    offsets=enc.pop("offset_mapping"); starts=[]; ends=[]
    for i,row in enumerate(examples):
        seq=enc.sequence_ids(i); ro=offsets[i].tolist(); cls_pos=(enc["input_ids"][i]==QA_TOKENIZER.cls_token_id).nonzero(); cls=int(cls_pos[0].item()) if len(cls_pos) else 0
        if row["answer_text"] is None:
            starts.append(cls); ends.append(cls); continue
        a0=int(row["answer_start"]); a1=a0+len(row["answer_text"]); ctx=[j for j,sid in enumerate(seq) if sid==1]; cs,ce=ctx[0],ctx[-1]
        if ro[cs][0]>a0 or ro[ce][1]<a1:
            starts.append(cls); ends.append(cls); continue
        while cs<=ce and ro[cs][0]<=a0: cs+=1
        while ce>=0 and ro[ce][1]>=a1: ce-=1
        starts.append(cs-1); ends.append(ce+1)
    enc["start_positions"]=torch.tensor(starts,dtype=torch.long); enc["end_positions"]=torch.tensor(ends,dtype=torch.long)
    return enc

def dev(batch): return {k:v.to(DEVICE) for k,v in batch.items()}
QA_POSITIVE_BATCH=dev(prepare_qa_batch(qa_train_rows)); QA_AUGMENTED_BATCH=dev(prepare_qa_batch(QA_TRAIN_AUGMENTED)); QA_VALIDATION_BATCH=dev(prepare_qa_batch(qa_validation_rows))
trainable=[p for p in QA_MODEL.parameters() if p.requires_grad]
opt=AdamW(trainable,lr=QA_LR,weight_decay=0.01)
QA_HISTORY=[]; QA_LOSSES=[]; QA_SPAN_WARMUP_EPOCHS=10
for epoch in range(QA_SPAN_WARMUP_EPOCHS):
    QA_MODEL.train(); opt.zero_grad(set_to_none=True); out=QA_MODEL(**QA_POSITIVE_BATCH); loss=out.loss
    if not torch.isfinite(loss): raise RuntimeError("Non-finite QA span warm-up loss")
    loss.backward(); torch.nn.utils.clip_grad_norm_(trainable,1.0); opt.step(); value=float(loss.detach().cpu()); QA_LOSSES.append(value); print(f"QA span-warmup epoch={epoch+1:02d} loss={value:.4f}")
QA_MAX_FINETUNE_EPOCHS=30; QA_PATIENCE=6; best_validation_loss=float("inf"); best_qa_state=None; best_qa_epoch=0; stale=0
for epoch in range(QA_MAX_FINETUNE_EPOCHS):
    QA_MODEL.train(); opt.zero_grad(set_to_none=True); out=QA_MODEL(**QA_AUGMENTED_BATCH); loss=out.loss
    if not torch.isfinite(loss): raise RuntimeError("Non-finite QA fine-tuning loss")
    loss.backward(); torch.nn.utils.clip_grad_norm_(trainable,1.0); opt.step()
    QA_MODEL.eval()
    with torch.no_grad(): vloss=float(QA_MODEL(**QA_VALIDATION_BATCH).loss.detach().cpu())
    t=float(loss.detach().cpu()); QA_LOSSES.append(t); QA_HISTORY.append({"epoch":epoch+1,"train_loss":t,"validation_loss":vloss}); print(f"QA finetune epoch={epoch+1:02d} train_loss={t:.4f} val_loss={vloss:.4f}")
    if vloss<best_validation_loss-1e-5:
        best_validation_loss=vloss; best_qa_epoch=epoch+1; stale=0; best_qa_state={n:p.detach().cpu().clone() for n,p in QA_MODEL.named_parameters() if p.requires_grad}
    else: stale+=1
    if stale>=QA_PATIENCE: print("QA early stopping:",{"epoch":epoch+1,"best_epoch":best_qa_epoch,"best_validation_loss":best_validation_loss}); break
if best_qa_state is None: raise RuntimeError("QA best checkpoint was not captured")
with torch.no_grad():
    for n,p in QA_MODEL.named_parameters():
        if n in best_qa_state: p.copy_(best_qa_state[n].to(p.device))
QA_STEPS=QA_SPAN_WARMUP_EPOCHS+len(QA_HISTORY)
print("QA_TRAINING=PASS",{"training_mode":QA_TRAINING_MODE,"span_warmup_epochs":QA_SPAN_WARMUP_EPOCHS,"finetune_epochs_run":len(QA_HISTORY),"total_optimiser_steps":QA_STEPS,"best_finetune_epoch":best_qa_epoch,"best_validation_loss":best_validation_loss,"selection_split":"official_validation_answerable_only","test_leakage":False})

QA_AUGMENTATION = {'official_train_answerable': 6, 'derived_train_no_answer': 2, 'official_validation_answerable': 2, 'derived_validation_no_answer': 2, 'test_used_for_training_or_tuning': False}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


QA span-warmup epoch=01 loss=3.8078
QA span-warmup epoch=02 loss=3.6729
QA span-warmup epoch=03 loss=3.4921
QA span-warmup epoch=04 loss=3.4202
QA span-warmup epoch=05 loss=3.2109
QA span-warmup epoch=06 loss=3.0914
QA span-warmup epoch=07 loss=2.9319
QA span-warmup epoch=08 loss=2.7341
QA span-warmup epoch=09 loss=2.5584
QA span-warmup epoch=10 loss=2.3765
QA finetune epoch=01 train_loss=2.6798 val_loss=2.9500
QA finetune epoch=02 train_loss=2.5567 val_loss=2.9139
QA finetune epoch=03 train_loss=2.3871 val_loss=2.8793
QA finetune epoch=04 train_loss=2.2343 val_loss=2.8477
QA finetune epoch=05 train_loss=2.0503 val_loss=2.8199
QA finetune epoch=06 train_loss=1.9192 val_loss=2.7976
QA finetune epoch=07 train_loss=1.8208 val_loss=2.7829
QA finetune epoch=08 train_loss=1.7332 val_loss=2.7798
QA finetune epoch=09 train_loss=1.6254 val_loss=2.7918
QA finetune epoch=10 train_loss=1.5466 val_loss=2.8195
QA finetune epoch=11 train_loss=1.3175 val_loss=2.8649
QA finetune epoch=12 train_loss=1.3

In [24]:
# Boundary and honest no-answer unit checks from the course contract.
context = "الخدمة متاحة في الرياض"
offsets = [None, (0,6), (7,12), (13,15), (16,22)]

valid_span = best_span(
    [0.0,0.1,0.1,0.2,4.0],
    [0.0,0.1,0.1,0.2,4.5],
    offsets,
    context,
)

null_span = best_span(
    [5.0,1.0,2.0],
    [5.0,1.0,2.0],
    [None,(0,6),(7,12)],
    "الخدمة متاحة",
    null_threshold=0.0,
)

assert valid_span["answer"] == "الرياض"
assert null_span["answer"] is None

print("QA_VALID_SPAN=PASS")
print("QA_NO_ANSWER=PASS")

QA_VALID_SPAN=PASS
QA_NO_ANSWER=PASS


# DAY 3 — Gate C: Arabic & Search

## 10. T1 — Arabic Profile with CAMeL Tools + Golden Tests

CAMeL Tools is used exactly where it adds value: search-oriented Arabic normalisation. Conservative text is kept for model tasks unless measured evidence justifies stronger normalisation.

In [25]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_unicode,
    normalize_alef_ar,
    normalize_alef_maksura_ar,
)

PROFILE_VERSION = "1.0.0"

def camel_search_profile(text: str):
    display_text = text
    model_text = mask_pii(text)
    model_text = normalize_unicode(model_text, compatibility=False)
    model_text = model_text.replace("ـ", "")
    model_text = dediac_ar(model_text)
    model_text = normalize_alef_ar(model_text)
    model_text = normalize_alef_maksura_ar(model_text)
    model_text = " ".join(model_text.split())

    return {
        "display_text": display_text,
        "model_text": model_text,
        "profile": "search",
        "backend": "camel-tools==1.6.0",
        "profile_version": PROFILE_VERSION,
    }

GOLDEN_ARABIC_CASES = [
    ("إِدَارَةُ الحِساب", "ادارة الحساب"),
    ("على  الطـريق", "علي الطريق"),
    ("راسل test@example.com", "راسل [EMAIL]"),
]

for raw,expected in GOLDEN_ARABIC_CASES:
    actual = camel_search_profile(raw)["model_text"]
    print(raw, "→", actual)
    assert actual == expected

print("ARABIC_GOLDEN_TESTS=PASS")

إِدَارَةُ الحِساب → ادارة الحساب
على  الطـريق → علي الطريق
راسل test@example.com → راسل [EMAIL]
ARABIC_GOLDEN_TESTS=PASS


In [26]:
ARABIC_DF = pd.read_csv(
    COURSE_DIR / "data/sample/bayan_day3_arabic.csv"
)

ARABIC_DF["conservative"] = ARABIC_DF["text"].map(
    lambda x: prepare_bayan_text(x, "ar")["model_text"]
)
ARABIC_DF["search_profile"] = ARABIC_DF["text"].map(
    lambda x: camel_search_profile(x)["model_text"]
)

ARABIC_DF["conservative_fertility"] = ARABIC_DF["conservative"].map(
    lambda x: token_fertility(x, tokenise)
)
ARABIC_DF["search_fertility"] = ARABIC_DF["search_profile"].map(
    lambda x: token_fertility(x, tokenise)
)

display(
    ARABIC_DF.groupby("variant")[
        ["conservative_fertility","search_fertility"]
    ].agg(["count","mean","max"])
)

conservative_fertility                     search_fertility            \
                         count      mean       max            count      mean   
variant                                                                         
Arabizi                      2  1.500000  1.600000                2  1.500000   
Gulf                         9  2.420370  2.800000                9  2.348148   
MSA                          9  2.451852  3.333333                9  2.359259   

                   
              max  
variant            
Arabizi  1.600000  
Gulf     2.666667  
MSA      2.800000

## 11. T4 — Bilingual Semantic Search

Creates multilingual sentence embeddings, L2-normalises them, and builds a FAISS `IndexFlatIP` index.

In [27]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

CASES_DF = pd.read_csv(
    COURSE_DIR / "data/sample/bayan_day3_cases.csv"
)
QUERY_ROWS = [
    json.loads(line)
    for line in (
        COURSE_DIR / "data/sample/bayan_day3_queries.jsonl"
    ).read_text(encoding="utf-8").splitlines()
    if line.strip()
]

SEARCH_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RERANKER_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

SEARCH_MODEL = SentenceTransformer(
    SEARCH_MODEL_ID,
    device=str(DEVICE),
)
RERANKER = CrossEncoder(
    RERANKER_ID,
    device=str(DEVICE),
)

def search_text(row):
    text = f"{row['summary']} {row['resolution']}"
    if row["language"] == "ar":
        return camel_search_profile(text)["model_text"]
    return normalize_whitespace(unicodedata.normalize("NFC", text))

case_rows = CASES_DF.to_dict("records")
corpus_texts = [search_text(r) for r in case_rows]

corpus_vectors = SEARCH_MODEL.encode(
    corpus_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

assert np.allclose(
    np.linalg.norm(corpus_vectors, axis=1),
    1.0,
    atol=1e-5,
)

FAISS_INDEX = faiss.IndexFlatIP(
    corpus_vectors.shape[1]
)
FAISS_INDEX.add(corpus_vectors)

CASE_LOOKUP = {
    r["case_id"]: r
    for r in case_rows
}

assert FAISS_INDEX.ntotal == len(case_rows)
print("FAISS_INDEX=PASS", FAISS_INDEX.ntotal)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS_INDEX=PASS 24


## 12. T4 — Re-Ranking and Retrieval Metrics

Measures cross-encoder re-ranking, Recall@10, MRR@10, latency, and cross-lingual retrieval.

In [28]:
def prepare_search_query(query, language):
    if language == "ar":
        return camel_search_profile(query)["model_text"]
    return normalize_whitespace(
        unicodedata.normalize("NFC", query)
    )

def retrieve(query, language, k=10):
    q = prepare_search_query(query, language)
    qv = SEARCH_MODEL.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, ids = FAISS_INDEX.search(
        qv,
        min(k, len(case_rows)),
    )

    return [
        {
            "case_id": case_rows[i]["case_id"],
            "score": float(score),
        }
        for i,score in zip(ids[0],scores[0])
    ]

def rerank(query, language, candidate_count=10):
    initial = retrieve(
        query,
        language,
        k=candidate_count,
    )

    pairs = [
        (
            query,
            f"{CASE_LOOKUP[item['case_id']]['summary']} "
            f"{CASE_LOOKUP[item['case_id']]['resolution']}"
        )
        for item in initial
    ]

    started = time.perf_counter()
    scores = RERANKER.predict(
        pairs,
        show_progress_bar=False,
    )
    rerank_ms = (time.perf_counter()-started)*1000

    ranked = sorted(
        zip(
            [item["case_id"] for item in initial],
            scores,
        ),
        key=lambda x:float(x[1]),
        reverse=True,
    )

    return [x[0] for x in ranked], rerank_ms

search_eval_rows = []

for q in QUERY_ROWS:
    before = [
        item["case_id"]
        for item in retrieve(
            q["query"],
            q["language"],
            k=10,
        )
    ]
    after, rerank_ms = rerank(
        q["query"],
        q["language"],
        candidate_count=10,
    )

    search_eval_rows.append({
        **q,
        "before_ids": before,
        "after_ids": after,
        "rerank_ms": rerank_ms,
    })

SEARCH_EVAL = pd.DataFrame(search_eval_rows)

answerable = SEARCH_EVAL[
    SEARCH_EVAL["relevant_case_ids"].map(bool)
]

before_metrics = retrieval_metrics(
    answerable["before_ids"].tolist(),
    answerable["relevant_case_ids"].tolist(),
    k=10,
)

after_metrics = retrieval_metrics(
    answerable["after_ids"].tolist(),
    answerable["relevant_case_ids"].tolist(),
    k=10,
)

print("BEFORE:", before_metrics)
print("AFTER:", after_metrics)
print("RERANK_P95_MS:", float(np.percentile(answerable["rerank_ms"],95)))

BEFORE: {'recall@10': 1.0, 'mrr@10': 0.75, 'answerable_queries': 12, 'skipped_no_answer': 0}
AFTER: {'recall@10': 1.0, 'mrr@10': 0.8194444444444443, 'answerable_queries': 12, 'skipped_no_answer': 0}
RERANK_P95_MS: 481.9781955499934


In [29]:
cross_lingual_slice = answerable[
    answerable["retrieval_mode"] == "cross_lingual"
]
monolingual_slice = answerable[
    answerable["retrieval_mode"] == "monolingual"
]

CROSS_LINGUAL_REPORT = pd.DataFrame([
    {
        "slice":"monolingual",
        "n":len(monolingual_slice),
        **retrieval_metrics(
            monolingual_slice["after_ids"].tolist(),
            monolingual_slice["relevant_case_ids"].tolist(),
            k=10,
        ),
    },
    {
        "slice":"cross_lingual",
        "n":len(cross_lingual_slice),
        **retrieval_metrics(
            cross_lingual_slice["after_ids"].tolist(),
            cross_lingual_slice["relevant_case_ids"].tolist(),
            k=10,
        ),
    },
])

display(CROSS_LINGUAL_REPORT)

,slice,n,recall@10,mrr@10,answerable_queries,skipped_no_answer
0,monolingual,8,1.0,0.854167,8,0
1,cross_lingual,4,1.0,0.750000,4,0


## 13. T4 — Search No-Answer Threshold

Tunes the no-answer threshold on validation queries only.

In [30]:
validation_queries = [
    q for q in QUERY_ROWS
    if q["split"] == "validation"
]

validation_scores = []
validation_has_relevant = []

for q in validation_queries:
    result = retrieve(
        q["query"],
        q["language"],
        k=1,
    )
    validation_scores.append(
        result[0]["score"]
    )
    validation_has_relevant.append(
        bool(q["relevant_case_ids"])
    )

SEARCH_THRESHOLD_REPORT = tune_no_answer_threshold(
    validation_scores,
    validation_has_relevant,
)

SEARCH_NO_ANSWER_THRESHOLD = SEARCH_THRESHOLD_REPORT["threshold"]

print(SEARCH_THRESHOLD_REPORT)

{'threshold': 0.4804013669490814, 'validation_accuracy': 1.0}


### Project-Model Evaluation Evidence

The distributed predictions fixture is kept for the full slice/bootstrap/error-analysis exercise.  
This additional table uses **this project's own final classifier predictions** so the notebook also contains traceable model evidence from the trained artefact.

In [31]:
PROJECT_EVAL_ROWS = []

for task in ["topic", "sentiment"]:
    truth_col = f"{task}_truth"
    pred_col = f"{task}_prediction"

    labels = sorted(
        set(PROJECT_TEST_PREDICTIONS[truth_col])
        | set(PROJECT_TEST_PREDICTIONS[pred_col])
    )

    overall_f1 = f1_score(
        PROJECT_TEST_PREDICTIONS[truth_col],
        PROJECT_TEST_PREDICTIONS[pred_col],
        labels=labels,
        average="macro",
        zero_division=0,
    )

    PROJECT_EVAL_ROWS.append({
        "task": task,
        "dimension": "overall",
        "slice": "all",
        "n": len(PROJECT_TEST_PREDICTIONS),
        "macro_f1": float(overall_f1),
    })

    for language, part in PROJECT_TEST_PREDICTIONS.groupby("language"):
        slice_labels = sorted(
            set(part[truth_col]) | set(part[pred_col])
        )
        PROJECT_EVAL_ROWS.append({
            "task": task,
            "dimension": "language",
            "slice": language,
            "n": len(part),
            "macro_f1": float(
                f1_score(
                    part[truth_col],
                    part[pred_col],
                    labels=slice_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
        })

PROJECT_MODEL_EVAL = pd.DataFrame(PROJECT_EVAL_ROWS)
display(PROJECT_MODEL_EVAL)

print("PROJECT_MODEL_TRACEABLE_EVALUATION=PASS")

,task,dimension,slice,n,macro_f1
0,topic,overall,all,8,0.866667
1,topic,language,ar,4,0.666667
2,topic,language,en,4,1.000000
3,sentiment,overall,all,8,0.533333
4,sentiment,language,ar,4,0.733333
5,sentiment,language,en,4,0.600000


PROJECT_MODEL_TRACEABLE_EVALUATION=PASS


## 14. T5 — Evaluation Slices and Uncertainty

Reports metrics by language, variant, and length with sample sizes and bootstrap confidence intervals.

In [32]:
PREDICTIONS_DF = pd.read_csv(
    COURSE_DIR / "data/sample/bayan_day3_predictions.csv"
)

from sklearn.metrics import f1_score

def topic_macro_f1(y_true, y_pred):
    labels = sorted(
        set(y_true) | set(y_pred)
    )
    return f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

slice_rows = []
for prediction_col in ["prediction_a","prediction_b"]:
    for dimension in ["language","variant","length_bucket"]:
        for value,part in PREDICTIONS_DF.groupby(dimension):
            slice_rows.append({
                "model": prediction_col,
                "dimension": dimension,
                "slice": value,
                "n": len(part),
                "macro_f1": topic_macro_f1(
                    part["topic"].tolist(),
                    part[prediction_col].tolist(),
                ),
            })

SLICE_REPORT = pd.DataFrame(slice_rows)
display(SLICE_REPORT)

,model,dimension,slice,n,macro_f1
0,prediction_a,language,ar,24,0.756119
1,prediction_a,language,en,12,0.828571
2,prediction_a,variant,English,12,0.828571
3,prediction_a,variant,Gulf,12,0.676190
4,prediction_a,variant,MSA,12,0.833333
5,prediction_a,length_bucket,long,18,0.572368
6,prediction_a,length_bucket,short,18,0.721362
7,prediction_b,language,ar,24,0.758289
8,prediction_b,language,en,12,0.828571
9,prediction_b,variant,English,12,0.828571


In [33]:
# Bootstrap uncertainty for Gulf slice using the course helper.
gulf = PREDICTIONS_DF[
    PREDICTIONS_DF["variant"] == "Gulf"
].reset_index(drop=True)

gulf_ci = bootstrap_ci(
    gulf["topic"].tolist(),
    gulf["prediction_b"].tolist(),
    topic_macro_f1,
    n_boot=1000,
    seed=SEED,
)

print("GULF_BOOTSTRAP_CI =", gulf_ci)

GULF_BOOTSTRAP_CI = {'estimate': 0.6583333333333333, 'bootstrap_mean': 0.6057101520755933, 'ci_low': 0.30000000000000004, 'ci_high': 0.903146853146853}


## 15. T5 — Error Analysis

Builds an error taxonomy and records three prioritised fixes.

In [34]:
ERROR_ROWS = PREDICTIONS_DF[
    PREDICTIONS_DF["topic"]
    != PREDICTIONS_DF["prediction_b"]
].copy()

def error_tag(row):
    if row["variant"] == "Gulf":
        return "dialect_gap"
    if row["length_bucket"] == "long":
        return "truncation"
    return "class_confusion"

TAGGED_ERRORS = [
    {
        "example_id": row["example_id"],
        "taxonomy_tag": error_tag(row),
    }
    for _,row in ERROR_ROWS.iterrows()
]

if TAGGED_ERRORS:
    ERROR_TAXONOMY = summarise_taxonomy(
        TAGGED_ERRORS
    )
else:
    ERROR_TAXONOMY = []

TOP_3_FIXES = pd.DataFrame([
    {
        "priority":1,
        "problem":"Gulf/dialect gap",
        "fix":"Add measured dialect-aware routing/profile comparison",
        "verification":"Macro-F1 on Gulf slice + CI",
    },
    {
        "priority":2,
        "problem":"Long-input/truncation risk",
        "fix":"Re-audit length distribution and compare max_length buckets",
        "verification":"Long-slice Macro-F1 and truncation rate",
    },
    {
        "priority":3,
        "problem":"Topic class confusion",
        "fix":"Review hard examples and class-specific errors",
        "verification":"Per-class F1 + behavioural minimum-functionality tests",
    },
])

print("ERROR_TAXONOMY =", ERROR_TAXONOMY)
display(TOP_3_FIXES)
print("PUBLIC_FIXTURE_ERRORS_AVAILABLE =", len(ERROR_ROWS))

ERROR_TAXONOMY = [{'taxonomy_tag': 'dialect_gap', 'count': 4}, {'taxonomy_tag': 'class_confusion', 'count': 2}, {'taxonomy_tag': 'truncation', 'count': 2}]


,priority,problem,fix,verification
0,1,Gulf/dialect gap,Add measured dialect-aware routing/profile com...,Macro-F1 on Gulf slice + CI
1,2,Long-input/truncation risk,Re-audit length distribution and compare max_l...,Long-slice Macro-F1 and truncation rate
2,3,Topic class confusion,Review hard examples and class-specific errors,Per-class F1 + behavioural minimum-functionali...


PUBLIC_FIXTURE_ERRORS_AVAILABLE = 8


## 16. T5 — Behavioural Tests

Checks basic invariance and minimum-functionality cases.

In [35]:
def predict_topic_text(text):
    model = TOPIC_ARTIFACT["model"]
    model.eval()

    prepared = prepare_bayan_text(
        text,
        infer_language(text),
    )["model_text"]

    encoded = CLASSIFIER_TOKENIZER(
        prepared,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded = {
        k: v.to(DEVICE)
        for k, v in encoded.items()
    }

    with torch.no_grad():
        pred_id = int(
            model(**encoded)
            .logits[0]
            .argmax()
            .item()
        )

    return TOPIC_ARTIFACT["id2label"][pred_id]

INVARIANCE_CASES = [
    (
        "تعذر تسجيل الدخول إلى البوابة",
        "  تعذر   تسجيل الدخول إلى البوابة  ",
    ),
    (
        "The bus did not arrive on time",
        "The   bus did not arrive on time",
    ),
]

invariance_rows = []

for first, second in INVARIANCE_CASES:
    p1 = predict_topic_text(first)
    p2 = predict_topic_text(second)

    invariance_rows.append({
        "case": first,
        "prediction_a": p1,
        "prediction_b": p2,
        "passed": p1 == p2,
    })

MINIMUM_CASES = [
    (
        "أحتاج معرفة حالة طلب التصريح",
        "permit",
    ),
    (
        "The bus did not arrive on time",
        "transport",
    ),
    (
        "My clinic appointment was delayed",
        "health",
    ),
    (
        "The verification code did not arrive",
        "digital_service",
    ),
]

minimum_rows = []

for text, gold in MINIMUM_CASES:
    pred = predict_topic_text(text)

    minimum_rows.append({
        "case": text,
        "gold": gold,
        "prediction": pred,
        "passed": pred == gold,
    })

BEHAVIOURAL_INVARIANCE = behavioural_pass_rate(
    invariance_rows
)
BEHAVIOURAL_MINIMUM = behavioural_pass_rate(
    minimum_rows
)

print("INVARIANCE:", BEHAVIOURAL_INVARIANCE)
print("MINIMUM_FUNCTIONALITY:", BEHAVIOURAL_MINIMUM)
display(pd.DataFrame(minimum_rows))

INVARIANCE: {'passed': 2, 'total': 2, 'pass_rate': 1.0}
MINIMUM_FUNCTIONALITY: {'passed': 4, 'total': 4, 'pass_rate': 1.0}


,case,gold,prediction,passed
0,أحتاج معرفة حالة طلب التصريح,permit,permit,True
1,The bus did not arrive on time,transport,transport,True
2,My clinic appointment was delayed,health,health,True
3,The verification code did not arrive,digital_service,digital_service,True


# DAY 4 — Gate D: Optimise & Serve

## 17. T6 — PROJECT_ARTIFACT Benchmark Contract

This section benchmarks the **topic classifier trained earlier in this notebook**. It is not the generic systems-smoke checkpoint.

Warm-up is separate from measured repetitions, and results include p50/p95/p99, throughput, memory method, quality tax, and a rollback decision.

In [36]:
from bayan.benchmarking import (
    benchmark_callable,
    quality_tax,
    artifact_size_mb,
    assess_budget,
    benchmark_concurrent,
)
from bayan.serving import sha256_file
import copy

# Fair before/after optimisation comparison:
# PyTorch, ONNX FP32 and ONNX INT8 all run on CPU with identical weights/data.
PROJECT_MODEL_CPU = copy.deepcopy(
    TOPIC_ARTIFACT["model"]
).to("cpu").eval()

PROJECT_TOKENIZER = CLASSIFIER_TOKENIZER
PROJECT_VALIDATION_ROWS = validation_rows

BENCH_BATCH_SIZE = 4
WARMUP = 3
REPETITIONS = 30

encoded_batches_cpu = []

for start in range(
    0,
    len(PROJECT_VALIDATION_ROWS),
    BENCH_BATCH_SIZE,
):
    batch_rows = PROJECT_VALIDATION_ROWS[
        start:start + BENCH_BATCH_SIZE
    ]

    encoded = PROJECT_TOKENIZER(
        [r["text"] for r in batch_rows],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded_batches_cpu.append({
        "input_ids": encoded["input_ids"].cpu(),
        "attention_mask": encoded["attention_mask"].cpu(),
    })

def pytorch_project_logits():
    outputs = []

    with torch.inference_mode():
        for batch in encoded_batches_cpu:
            outputs.append(
                PROJECT_MODEL_CPU(**batch)
                .logits.detach().cpu().numpy()
            )

    return np.concatenate(outputs, axis=0)

baseline_logits = pytorch_project_logits()
baseline_predictions = baseline_logits.argmax(axis=1)

try:
    import psutil
    process = psutil.Process(os.getpid())
    memory_reader = lambda: process.memory_info().rss
    MEMORY_METHOD = "process RSS start and observed peak; approximate"
except Exception:
    memory_reader = None
    MEMORY_METHOD = "RSS unavailable"

PYTORCH_BENCHMARK = benchmark_callable(
    pytorch_project_logits,
    warmup=WARMUP,
    repetitions=REPETITIONS,
    items_per_call=len(PROJECT_VALIDATION_ROWS),
    memory_reader=memory_reader,
)

print("BENCHMARK_DEVICE = CPU for PyTorch/ONNX comparison")
print("PYTORCH_PROJECT_ARTIFACT =", PYTORCH_BENCHMARK)

BENCHMARK_DEVICE = CPU for PyTorch/ONNX comparison
PYTORCH_PROJECT_ARTIFACT = {'warmup': 3, 'repetitions': 30, 'items_per_call': 8, 'p50_ms': 211.0875025, 'p95_ms': 1027.2678429500002, 'p99_ms': 1375.4085029900002, 'mean_ms': 397.5204902, 'min_ms': 194.765107, 'max_ms': 1475.615645, 'throughput_items_s': 20.124748779553602, 'rss_start_mb': 3744.4296875, 'rss_peak_observed_mb': 3744.4296875, 'rss_observed_delta_mb': 0.0}


## 18. T6 — ONNX and INT8 Optimisation

Exports the project model to ONNX, verifies parity, and measures a dynamic INT8 attempt.

In [37]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_dynamic,
    QuantType,
    quant_pre_process,
)

ARTIFACT_DIR = Path("/content/bayan_optimisation")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FP32_ONNX = ARTIFACT_DIR / "bayan_topic_fp32.onnx"
FP32_PREPROCESSED = ARTIFACT_DIR / "bayan_topic_fp32_preprocessed.onnx"
INT8_ONNX = ARTIFACT_DIR / "bayan_topic_dynamic_int8.onnx"

class LogitsWrapper(torch.nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, input_ids, attention_mask):
        return self.inner(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).logits

wrapper = LogitsWrapper(PROJECT_MODEL_CPU).eval()

example_batch = encoded_batches_cpu[0]

torch.onnx.export(
    wrapper,
    (
        example_batch["input_ids"],
        example_batch["attention_mask"],
    ),
    str(FP32_ONNX),
    input_names=["input_ids","attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids":{0:"batch",1:"sequence"},
        "attention_mask":{0:"batch",1:"sequence"},
        "logits":{0:"batch"},
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)

onnx_model = onnx.load(str(FP32_ONNX))
onnx.checker.check_model(onnx_model)

quant_pre_process(
    str(FP32_ONNX),
    str(FP32_PREPROCESSED),
    skip_symbolic_shape=True,
)

print("ONNX_CHECKER=PASS")

/tmp/ipykernel_280/2740241020.py:31: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:212: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:120: TracerWarning: Converting a tensor to a Python boolean

ONNX_CHECKER=PASS


In [38]:
def ort_inputs(batch):
    return {
        "input_ids": batch["input_ids"].detach().cpu().numpy().astype(np.int64),
        "attention_mask": batch["attention_mask"].detach().cpu().numpy().astype(np.int64),
    }

fp32_session = ort.InferenceSession(
    str(FP32_ONNX),
    providers=["CPUExecutionProvider"],
)

def onnx_fp32_logits():
    return np.concatenate([
        fp32_session.run(
            ["logits"],
            ort_inputs(batch),
        )[0]
        for batch in encoded_batches_cpu
    ], axis=0)

fp32_logits = onnx_fp32_logits()
fp32_predictions = fp32_logits.argmax(axis=1)

FP32_PARITY = {
    "max_abs_logits_diff": float(
        np.max(np.abs(baseline_logits-fp32_logits))
    ),
    "prediction_agreement": float(
        np.mean(baseline_predictions==fp32_predictions)
    ),
}

assert FP32_PARITY["max_abs_logits_diff"] < 1e-3
assert FP32_PARITY["prediction_agreement"] == 1.0

ONNX_FP32_BENCHMARK = benchmark_callable(
    onnx_fp32_logits,
    warmup=WARMUP,
    repetitions=REPETITIONS,
    items_per_call=len(PROJECT_VALIDATION_ROWS),
    memory_reader=memory_reader,
)

print("ONNX_FP32_PARITY=PASS", FP32_PARITY)
print("ONNX_FP32_BENCHMARK =", ONNX_FP32_BENCHMARK)

ONNX_FP32_PARITY=PASS {'max_abs_logits_diff': 2.6226043701171875e-06, 'prediction_agreement': 1.0}
ONNX_FP32_BENCHMARK = {'warmup': 3, 'repetitions': 30, 'items_per_call': 8, 'p50_ms': 161.582377, 'p95_ms': 246.86073770000002, 'p99_ms': 252.2619489, 'mean_ms': 176.14824033333335, 'min_ms': 156.705533, 'max_ms': 253.534669, 'throughput_items_s': 45.416292464013466, 'rss_start_mb': 4505.703125, 'rss_peak_observed_mb': 4505.703125, 'rss_observed_delta_mb': 0.0}


In [39]:
INT8_STATUS = {
    "attempted": True,
    "available": False,
    "error": None,
}
INT8_SESSION = None
INT8_BENCHMARK = None
INT8_PARITY = None

try:
    quantize_dynamic(
        model_input=str(FP32_PREPROCESSED),
        model_output=str(INT8_ONNX),
        weight_type=QuantType.QInt8,
    )

    INT8_SESSION = ort.InferenceSession(
        str(INT8_ONNX),
        providers=["CPUExecutionProvider"],
    )

    def int8_logits_fn():
        return np.concatenate([
            INT8_SESSION.run(
                ["logits"],
                ort_inputs(batch),
            )[0]
            for batch in encoded_batches_cpu
        ], axis=0)

    int8_logits = int8_logits_fn()
    int8_predictions = int8_logits.argmax(axis=1)

    INT8_PARITY = {
        "max_abs_logits_diff": float(
            np.max(np.abs(baseline_logits-int8_logits))
        ),
        "prediction_agreement": float(
            np.mean(baseline_predictions==int8_predictions)
        ),
    }

    INT8_BENCHMARK = benchmark_callable(
        int8_logits_fn,
        warmup=WARMUP,
        repetitions=REPETITIONS,
        items_per_call=len(PROJECT_VALIDATION_ROWS),
        memory_reader=memory_reader,
    )

    INT8_STATUS["available"] = True
    print("INT8_ATTEMPT=PASS")

except Exception as exc:
    INT8_STATUS["error"] = f"{type(exc).__name__}: {exc}"
    print("INT8_ATTEMPT=DOCUMENTED_UNSUPPORTED", INT8_STATUS["error"])

assert INT8_STATUS["attempted"]
assert INT8_STATUS["available"] or INT8_STATUS["error"]

INT8_ATTEMPT=PASS


## 19. T6 — Quality Tax and Optimisation Decision

Keeps an optimised artefact only when quality and performance stay within the selected budget.

In [40]:
topic_labels = TOPIC_ARTIFACT["labels"]
validation_truth = [r["topic"] for r in PROJECT_VALIDATION_ROWS]

baseline_label_predictions = [
    TOPIC_ARTIFACT["id2label"][int(i)]
    for i in baseline_predictions
]
fp32_label_predictions = [
    TOPIC_ARTIFACT["id2label"][int(i)]
    for i in fp32_predictions
]

baseline_quality = f1_score(
    validation_truth,
    baseline_label_predictions,
    labels=topic_labels,
    average="macro",
    zero_division=0,
)
fp32_quality = f1_score(
    validation_truth,
    fp32_label_predictions,
    labels=topic_labels,
    average="macro",
    zero_division=0,
)

FP32_QUALITY_TAX = quality_tax(
    baseline_quality,
    fp32_quality,
)

PERFORMANCE_BUDGET = {
    "max_p95_ms": PYTORCH_BENCHMARK["p95_ms"],
    "max_quality_tax": 0.01,
    "min_throughput_items_s": 0.0,
}

FP32_BUDGET = assess_budget(
    ONNX_FP32_BENCHMARK,
    quality_tax_value=FP32_QUALITY_TAX,
    max_p95_ms=PERFORMANCE_BUDGET["max_p95_ms"],
    max_quality_tax=PERFORMANCE_BUDGET["max_quality_tax"],
    min_throughput_items_s=PERFORMANCE_BUDGET["min_throughput_items_s"],
)

if INT8_STATUS["available"]:
    int8_label_predictions = [
        TOPIC_ARTIFACT["id2label"][int(i)]
        for i in int8_predictions
    ]
    int8_quality = f1_score(
        validation_truth,
        int8_label_predictions,
        labels=topic_labels,
        average="macro",
        zero_division=0,
    )
    INT8_QUALITY_TAX = quality_tax(
        baseline_quality,
        int8_quality,
    )
    INT8_BUDGET = assess_budget(
        INT8_BENCHMARK,
        quality_tax_value=INT8_QUALITY_TAX,
        max_p95_ms=PERFORMANCE_BUDGET["max_p95_ms"],
        max_quality_tax=PERFORMANCE_BUDGET["max_quality_tax"],
        min_throughput_items_s=PERFORMANCE_BUDGET["min_throughput_items_s"],
    )
else:
    INT8_QUALITY_TAX = None
    INT8_BUDGET = None

if INT8_STATUS["available"] and INT8_BUDGET["budget_met"]:
    OPTIMISATION_DECISION = "ADOPT_INT8"
elif FP32_BUDGET["budget_met"]:
    OPTIMISATION_DECISION = "ADOPT_ONNX_FP32"
else:
    OPTIMISATION_DECISION = "ROLLBACK_TO_PYTORCH_FP32"

print({
    "baseline_quality": baseline_quality,
    "fp32_quality_tax": FP32_QUALITY_TAX,
    "fp32_budget": FP32_BUDGET,
    "int8_quality_tax": INT8_QUALITY_TAX,
    "int8_budget": INT8_BUDGET,
    "decision": OPTIMISATION_DECISION,
})

ARTIFACT_SIZES_MB = {
    "project_saved_model": artifact_size_mb(PROJECT_ARTIFACT_DIR),
    "onnx_fp32": artifact_size_mb(FP32_ONNX),
    "onnx_int8": (
        artifact_size_mb(INT8_ONNX)
        if INT8_STATUS["available"] and INT8_ONNX.exists()
        else None
    ),
}

print("ARTIFACT_SIZES_MB =", ARTIFACT_SIZES_MB)

{'baseline_quality': 1.0, 'fp32_quality_tax': 0.0, 'fp32_budget': {'latency_ok': True, 'quality_ok': True, 'throughput_ok': True, 'budget_met': True}, 'int8_quality_tax': 0.5654761904761905, 'int8_budget': {'latency_ok': True, 'quality_ok': False, 'throughput_ok': True, 'budget_met': False}, 'decision': 'ADOPT_ONNX_FP32'}
ARTIFACT_SIZES_MB = {'project_saved_model': 519.0318756103516, 'onnx_fp32': 516.358226776123, 'onnx_int8': 129.4597225189209}


## 20. T6 — Unified API

Returns topic, sentiment, entities, optional QA, search results, model information, and latency.

In [41]:
def predict_sequence_artifact(artifact, text):
    model=artifact["model"]; model.eval(); encoded=CLASSIFIER_TOKENIZER(prepare_bayan_text(text,infer_language(text))["model_text"],truncation=True,max_length=MAX_LENGTH,return_tensors="pt"); encoded={k:v.to(DEVICE) for k,v in encoded.items()}
    with torch.no_grad(): idx=int(model(**encoded).logits[0].argmax().item())
    return artifact["id2label"][idx]

SERVING_ONNX_SESSION=None
if OPTIMISATION_DECISION=="ADOPT_ONNX_FP32":
    serving_options=ort.SessionOptions(); serving_options.intra_op_num_threads=1; serving_options.inter_op_num_threads=1; serving_options.execution_mode=ort.ExecutionMode.ORT_SEQUENTIAL
    SERVING_ONNX_SESSION=ort.InferenceSession(str(FP32_ONNX),sess_options=serving_options,providers=["CPUExecutionProvider"])

def predict_topic_for_service(text):
    prepared=prepare_bayan_text(text,infer_language(text))["model_text"]
    if SERVING_ONNX_SESSION is not None:
        encoded=CLASSIFIER_TOKENIZER(prepared,truncation=True,max_length=MAX_LENGTH,return_tensors="np")
        logits=SERVING_ONNX_SESSION.run(["logits"],{"input_ids":encoded["input_ids"].astype(np.int64),"attention_mask":encoded["attention_mask"].astype(np.int64)})[0]
        return TOPIC_ARTIFACT["id2label"][int(np.argmax(logits[0]))]
    return predict_sequence_artifact(TOPIC_ARTIFACT,text)

def predict_sentiment_for_service(text):
    if SENTIMENT_SERVING_BACKEND=="tfidf_baseline": return str(SENTIMENT_ARTIFACT["baseline"].predict([text])[0])
    return predict_sequence_artifact(SENTIMENT_ARTIFACT,text)

def extract_entities(text):
    words = text.split()

    if not words:
        return []

    word_logits = ner_word_logits_from_tokens(words)
    pred_ids = bio_viterbi_decode(word_logits)
    labels = [
        NER_ID2LABEL[int(pred_id)]
        for pred_id in pred_ids
    ]

    entities = []
    current = None

    for index, (word, label) in enumerate(
        zip(words, labels)
    ):
        prefix, entity_type = _bio_parts(label)

        if prefix == "B":
            if current is not None:
                entities.append(current)

            current = {
                "text": word,
                "label": entity_type,
                "start_word": index,
                "end_word": index,
            }

        elif prefix == "I" and current is not None:
            current["text"] += " " + word
            current["end_word"] = index

        else:
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    return entities


def qa_raw_result(question, context, null_threshold=0.0):
    encoded = QA_TOKENIZER(
        question,
        context,
        truncation="only_second",
        max_length=QA_MAX_LENGTH,
        return_offsets_mapping=True,
        return_tensors="pt",
    )

    sequence_ids = encoded.sequence_ids(0)
    raw_offsets = encoded.pop(
        "offset_mapping"
    )[0].tolist()

    offsets = [
        tuple(offset)
        if sequence_ids[i] == 1
        else None
        for i, offset in enumerate(raw_offsets)
    ]

    inputs = {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }

    QA_MODEL.eval()

    with torch.no_grad():
        output = QA_MODEL(**inputs)

    return best_span(
        output.start_logits[0].cpu().tolist(),
        output.end_logits[0].cpu().tolist(),
        offsets,
        context,
        null_threshold=null_threshold,
    )

def qa_inference(question, context, null_threshold=None):
    if null_threshold is None:
        null_threshold = QA_NULL_THRESHOLD

    return qa_raw_result(
        question,
        context,
        null_threshold=float(null_threshold),
    )

def semantic_search_service(text, language, k=3):
    best = retrieve(text, language, k=k)

    if best and best[0]["score"] < SEARCH_NO_ANSWER_THRESHOLD:
        return []

    readable = []
    for item in best:
        case = CASE_LOOKUP[item["case_id"]]
        readable.append({
            "case_id": item["case_id"],
            "score": item["score"],
            "language": case["language"],
            "summary": case["summary"],
            "resolution": case["resolution"],
        })

    return readable

def bayan_classify(text):
    clean = validate_request_text(text, max_chars=1000)
    return {
        "language": infer_language(clean),
        "topic": predict_topic_for_service(clean),
        "sentiment": predict_sentiment_for_service(clean),
        "sentiment_backend": SENTIMENT_SERVING_BACKEND,
        "topic_serving_engine": ("onnxruntime_fp32" if SERVING_ONNX_SESSION is not None else "pytorch"),
    }

def bayan_analyse(text, question=None, context=None):
    started = time.perf_counter()

    clean = validate_request_text(text, max_chars=1000)
    language = infer_language(clean)
    prepared = prepare_bayan_text(clean, language)

    response = {
        "language": language,
        "safe_text": mask_pii(prepared["display_text"]),
        "topic": predict_topic_for_service(clean),
        "sentiment": predict_sentiment_for_service(clean),
        "entities": extract_entities(clean),
        "qa": None,
        "similar_cases": semantic_search_service(
            prepared["model_text"],
            language,
            k=3,
        ),
        "preprocessing_version": PREPROCESSING_VERSION,
        "models": {
            "topic": CLASSIFIER_MODEL_ID,
            "sentiment": (
                CLASSIFIER_MODEL_ID
                if SENTIMENT_SERVING_BACKEND == "transformer"
                else "TF-IDF char_wb + LinearSVC"
            ),
            "ner": CLASSIFIER_MODEL_ID,
            "qa": CLASSIFIER_MODEL_ID,
            "search": SEARCH_MODEL_ID,
            "reranker": RERANKER_ID,
        },
    }

    if question is not None and context is not None:
        response["qa"] = qa_inference(question, context)

    response["latency_ms"] = round(
        (time.perf_counter() - started) * 1000,
        3,
    )
    return response

display(
    bayan_analyse(
        "أحتاج معرفة حالة طلب التصريح"
    )
)

{'language': 'ar',
 'safe_text': 'أحتاج معرفة حالة طلب التصريح',
 'topic': 'permit',
 'sentiment': 'neutral',
 'entities': [{'text': 'طلب التصريح',
   'label': 'SERVICE',
   'start_word': 3,
   'end_word': 4}],
 'qa': None,
 'similar_cases': [{'case_id': 'AR-010',
   'score': 0.5466217994689941,
   'language': 'ar',
   'summary': 'رُفض المستند المرفق بطلب التصريح',
   'resolution': 'تم توضيح صيغة المستند وإعادة فتح الطلب للرفع'},
  {'case_id': 'AR-011',
   'score': 0.5033593773841858,
   'language': 'ar',
   'summary': 'طلب التصريح واقف عند المراجعة من أسبوع',
   'resolution': 'تم تصعيد الطلب وتحديث الحالة في اليوم التالي'},
  {'case_id': 'EN-010',
   'score': 0.38467392325401306,
   'language': 'en',
   'summary': 'The permit request rejected the uploaded document',
   'resolution': 'The required format was explained and upload was reopened'}],
 'preprocessing_version': 'shouq-bayan-preprocess-v1.0',
 'models': {'topic': 'distilbert/distilbert-base-multilingual-cased',
  'sentiment': 

### QA Validation Smoke

This records the QA model's actual validation behaviour without treating the tiny public fixture as production-quality evidence.

### QA no-answer calibration

The public QA fixture is very small. The QA head is first warmed up on the official answerable training examples so it learns answer boundaries before limited no-answer fine-tuning. A small number of same-language cross-context no-answer examples are derived from the official training split only.

The null threshold is selected on validation only using a balanced utility that combines answerable token-F1 with no-answer accuracy. The official test split remains untouched until final evaluation.

In [42]:
def _qa_token_f1(gold,prediction):
    if gold is None or prediction is None: return 1.0 if gold is None and prediction is None else 0.0
    gt=str(gold).split(); pt=str(prediction).split()
    if not gt or not pt: return float(gt==pt)
    overlap=sum((Counter(gt)&Counter(pt)).values())
    if overlap==0:return 0.0
    p=overlap/len(pt); r=overlap/len(gt); return 2*p*r/(p+r)

def qa_margin_record(row):
    result=qa_raw_result(row["question"],row["context"],null_threshold=1e9)
    return {"id":row["id"],"gold":row["answer_text"],"span_prediction":result.get("answer"),"null_margin":float(result.get("null_margin",result.get("margin",float("inf")))),"is_no_answer":row["answer_text"] is None}
QA_THRESHOLD_VALIDATION_ROWS=qa_validation_rows+QA_VALIDATION_NEGATIVES
QA_THRESHOLD_VALIDATION=[qa_margin_record(r) for r in QA_THRESHOLD_VALIDATION_ROWS]

def _threshold_candidates(records):
    m=sorted({float(r["null_margin"]) for r in records if np.isfinite(r["null_margin"])})
    if not m:return [0.0]
    c=[m[0]-1.0,m[-1]+1.0,*m]
    c += [(a+b)/2.0 for a,b in zip(m[:-1],m[1:])]
    return sorted(set(c))

def evaluate_qa_threshold(records,threshold):
    scores=[]; em=0; at=0; nac=0; nat=0
    for row in records:
        null=row["null_margin"]>threshold
        if row["is_no_answer"]:
            nat+=1; nac+=int(null); continue
        at+=1; pred=None if null else row["span_prediction"]; scores.append(_qa_token_f1(row["gold"],pred)); em+=int(pred==row["gold"])
    tf1=float(np.mean(scores)) if scores else 0.0; emr=em/at if at else 0.0; naa=nac/nat if nat else 0.0; utility=0.5*tf1+0.5*naa
    return {"threshold":float(threshold),"answerable_token_f1":tf1,"answerable_exact_match":float(emr),"no_answer_accuracy":float(naa),"balanced_utility":float(utility)}
QA_THRESHOLD_SWEEP=[evaluate_qa_threshold(QA_THRESHOLD_VALIDATION,t) for t in _threshold_candidates(QA_THRESHOLD_VALIDATION)]
QA_THRESHOLD_REPORT=max(QA_THRESHOLD_SWEEP,key=lambda r:(r["balanced_utility"],r["answerable_token_f1"],r["no_answer_accuracy"],-abs(r["threshold"])))
QA_NULL_THRESHOLD=float(QA_THRESHOLD_REPORT["threshold"])
print("QA_NULL_THRESHOLD_VALIDATION_ONLY =",QA_THRESHOLD_REPORT); display(pd.DataFrame(QA_THRESHOLD_SWEEP).sort_values(["balanced_utility","answerable_token_f1"],ascending=False))

def qa_validation_smoke(rows):
    rec=[]
    for row in rows:
        result=qa_inference(row["question"],row["context"]); pred=result["answer"]; gold=row["answer_text"]
        rec.append({"example_id":row["id"],"gold":gold,"prediction":pred,"exact_match":pred==gold,"token_f1":_qa_token_f1(gold,pred),"gold_is_null":gold is None,"predicted_is_null":pred is None})
    frame=pd.DataFrame(rec)
    return frame,{"n":len(frame),"exact_match":float(frame["exact_match"].mean()) if len(frame) else 0.0,"mean_token_f1":float(frame["token_f1"].mean()) if len(frame) else 0.0,"no_answer_correct":int((frame["gold_is_null"]&frame["predicted_is_null"]).sum()),"no_answer_total":int(frame["gold_is_null"].sum())}
QA_VALIDATION_SMOKE,QA_VALIDATION_METRICS=qa_validation_smoke(qa_validation_rows); display(QA_VALIDATION_SMOKE); print("QA_VALIDATION_MEASURED_SMOKE =",QA_VALIDATION_METRICS)
QA_VALIDATION_NO_ANSWER_SMOKE,QA_VALIDATION_NO_ANSWER_METRICS=qa_validation_smoke(QA_VALIDATION_NEGATIVES); display(QA_VALIDATION_NO_ANSWER_SMOKE); print("QA_VALIDATION_NO_ANSWER_DERIVED =",QA_VALIDATION_NO_ANSWER_METRICS)
QA_TEST_NO_ANSWER_RESULTS=[]
for row in qa_test_rows:
    result=qa_inference(row["question"],row["context"]); QA_TEST_NO_ANSWER_RESULTS.append({"id":row["id"],"prediction":result["answer"],"correct_no_answer":row["answer_text"] is None and result["answer"] is None,"reason":result.get("reason"),"margin":result.get("margin",result.get("null_margin"))})
QA_TEST_NO_ANSWER_METRICS={"n":len(QA_TEST_NO_ANSWER_RESULTS),"correct":sum(int(r["correct_no_answer"]) for r in QA_TEST_NO_ANSWER_RESULTS)}; QA_TEST_NO_ANSWER_METRICS["accuracy"]=QA_TEST_NO_ANSWER_METRICS["correct"]/QA_TEST_NO_ANSWER_METRICS["n"] if QA_TEST_NO_ANSWER_METRICS["n"] else 0.0
print("QA_TEST_NO_ANSWER_MEASURED_SMOKE =",QA_TEST_NO_ANSWER_METRICS); display(pd.DataFrame(QA_TEST_NO_ANSWER_RESULTS))
train_ids={r["id"] for r in QA_TRAIN_AUGMENTED}; assert all(r["id"] not in train_ids for r in qa_test_rows); print("QA_TEST_LEAKAGE_CHECK=PASS")

QA_NULL_THRESHOLD_VALIDATION_ONLY = {'threshold': -2.9909180402755737, 'answerable_token_f1': 0.0, 'answerable_exact_match': 0.0, 'no_answer_accuracy': 1.0, 'balanced_utility': 0.5}


,threshold,answerable_token_f1,answerable_exact_match,no_answer_accuracy,balanced_utility
0,-2.990918,0.000000,0.0,1.0,0.500000
3,-1.712868,0.222222,0.0,0.5,0.361111
4,-1.382877,0.222222,0.0,0.5,0.361111
1,-1.990918,0.000000,0.0,0.5,0.250000
2,-1.851893,0.000000,0.0,0.5,0.250000
5,-1.052886,0.222222,0.0,0.0,0.111111
6,-0.907326,0.222222,0.0,0.0,0.111111
7,-0.761766,0.222222,0.0,0.0,0.111111
8,0.238234,0.222222,0.0,0.0,0.111111


,example_id,gold,prediction,exact_match,token_f1,gold_is_null,predicted_is_null
0,Q-007,عبر التطبيق أو مركز الاتصال,None,False,0.0,False,True
1,Q-008,PDF,None,False,0.0,False,True


QA_VALIDATION_MEASURED_SMOKE = {'n': 2, 'exact_match': 0.0, 'mean_token_f1': 0.0, 'no_answer_correct': 0, 'no_answer_total': 0}


,example_id,gold,prediction,exact_match,token_f1,gold_is_null,predicted_is_null
0,Q-007__VAL_NEG__Q-001,None,None,True,1.0,True,True
1,Q-008__VAL_NEG__Q-002,None,None,True,1.0,True,True


QA_VALIDATION_NO_ANSWER_DERIVED = {'n': 2, 'exact_match': 1.0, 'mean_token_f1': 1.0, 'no_answer_correct': 2, 'no_answer_total': 2}
QA_TEST_NO_ANSWER_MEASURED_SMOKE = {'n': 2, 'correct': 1, 'accuracy': 0.5}


,id,prediction,correct_no_answer,reason,margin
0,Q-009,الأحد,False,None,-4.282570
1,Q-010,None,True,no_answer_in_context,0.060722


QA_TEST_LEAKAGE_CHECK=PASS


In [43]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient
from typing import Optional

app = FastAPI(
    title="Bayan Unified NLP API",
    version="1.0.0",
)

class AnalyseRequest(BaseModel):
    text: str = Field(min_length=1, max_length=1000)
    question: Optional[str] = None
    context: Optional[str] = None

@app.get("/health")
def health():
    return {
        "status":"ready",
        "preprocessing_version":PREPROCESSING_VERSION,
        "project_artifact":str(PROJECT_ARTIFACT_DIR),
        "optimisation_decision":OPTIMISATION_DECISION,
    }


@app.post("/v1/classify")
def classify(request: AnalyseRequest):
    try:
        return bayan_classify(request.text)
    except ValueError as exc:
        raise HTTPException(
            status_code=422,
            detail=str(exc),
        )

@app.post("/v1/analyse")
def analyse(request: AnalyseRequest):
    try:
        return bayan_analyse(
            request.text,
            request.question,
            request.context,
        )
    except ValueError as exc:
        raise HTTPException(
            status_code=422,
            detail=str(exc),
        )

with TestClient(app) as client:
    health_response = client.get("/health")
    classify_response = client.post(
        "/v1/classify",
        json={"text":"أحتاج معرفة حالة طلب التصريح"},
    )
    ar_response = client.post(
        "/v1/analyse",
        json={"text":"تأخر التصريح في الرياض"},
    )
    en_response = client.post(
        "/v1/analyse",
        json={"text":"The verification code did not arrive"},
    )
    invalid_response = client.post(
        "/v1/analyse",
        json={"text":"   "},
    )

assert health_response.status_code == 200
assert classify_response.status_code == 200
assert ar_response.status_code == 200
assert en_response.status_code == 200
assert invalid_response.status_code == 422

print("UNIFIED_API_TESTS=PASS")

UNIFIED_API_TESTS=PASS


## 21. T6 — Concurrent HTTP Benchmark

Measures API latency under concurrent requests and reports p50, p95, and p99.

In [44]:
def concurrent_classifier_http_benchmark(payloads,concurrency=16,requests=64):
    import threading
    payloads=list(payloads); counter={"i":0}; lock=threading.Lock()
    with TestClient(app) as bench_client:
        def workload():
            with lock:
                i=counter["i"]; counter["i"]+=1
            response=bench_client.post("/v1/classify",json={"text":payloads[i%len(payloads)]})
            if response.status_code!=200: raise RuntimeError(f"HTTP {response.status_code}: {response.text}")
            return response.json()
        return benchmark_concurrent(workload,warmup_requests=16,requests=requests,concurrency=concurrency)
HTTP_SERVING_ENGINE="onnxruntime_fp32" if SERVING_ONNX_SESSION is not None else "pytorch"
HTTP_BENCHMARK=concurrent_classifier_http_benchmark([r["text"] for r in validation_rows],concurrency=16,requests=64)
HTTP_THREAD_SWEEP=[]; HTTP_SELECTED_TORCH_THREADS=None
print("HTTP_SERVING_ENGINE =",HTTP_SERVING_ENGINE); print("CLASSIFIER_HTTP_BENCHMARK =",HTTP_BENCHMARK)

HTTP_SERVING_ENGINE = onnxruntime_fp32
CLASSIFIER_HTTP_BENCHMARK = {'repetitions': 64, 'items_per_call': 1, 'p50_ms': 1017.3553300000001, 'p95_ms': 1182.8438545, 'p99_ms': 1454.39000931, 'mean_ms': 1008.481545953125, 'min_ms': 549.478772, 'max_ms': 1476.200712, 'throughput_items_s': 15.48076173290254, 'warmup_requests': 16, 'requests': 64, 'concurrency': 16, 'wall_time_s': 4.134163493000187, 'throughput_requests_s': 15.48076173290254}


# T7 — Required Measured Extension: Dialect Router

The selected extension is a bounded Gulf-dialect router. It does not make eligibility or policy decisions. It only provides a routing signal before downstream processing.

The extension is compared with a baseline and reports both benefit and latency cost.

In [45]:
import re

GULF_CUES = {
    "وش","وين","أبي","ابي","الحين",
    "واقف","ودي","كذا","لسى","مره","مرررة",
}

def dialect_router(text):
    tokens = set(
        re.findall(r"[؀-ۿ]+", text)
    )
    hits = sorted(tokens & GULF_CUES)
    return {
        "route":"Gulf" if hits else "standard_or_other",
        "hits":hits,
    }

dialect_eval = ARABIC_DF[
    ARABIC_DF["variant"].isin(["Gulf","MSA"])
].copy()

dialect_eval["gold"] = dialect_eval["variant"].map(
    lambda x:"Gulf" if x=="Gulf" else "standard_or_other"
)
dialect_eval["baseline"] = "standard_or_other"
dialect_eval["extension"] = dialect_eval["text"].map(
    lambda x:dialect_router(x)["route"]
)

BASELINE_EXT_ACC = float(
    (dialect_eval["baseline"]==dialect_eval["gold"]).mean()
)
EXTENSION_ACC = float(
    (dialect_eval["extension"]==dialect_eval["gold"]).mean()
)

def baseline_extension_call(text):
    return "standard_or_other"

def measured_extension_call(text):
    return dialect_router(text)["route"]

BASE_EXTENSION_BENCH = benchmark_callable(
    lambda: [baseline_extension_call(x) for x in dialect_eval["text"]],
    warmup=3,
    repetitions=30,
    items_per_call=len(dialect_eval),
)

DIALECT_EXTENSION_BENCH = benchmark_callable(
    lambda: [measured_extension_call(x) for x in dialect_eval["text"]],
    warmup=3,
    repetitions=30,
    items_per_call=len(dialect_eval),
)

EXTENSION_EVIDENCE = {
    "baseline_accuracy": BASELINE_EXT_ACC,
    "extension_accuracy": EXTENSION_ACC,
    "benefit": EXTENSION_ACC-BASELINE_EXT_ACC,
    "baseline_p95_ms": BASE_EXTENSION_BENCH["p95_ms"],
    "extension_p95_ms": DIALECT_EXTENSION_BENCH["p95_ms"],
    "cost_p95_ms": DIALECT_EXTENSION_BENCH["p95_ms"]-BASE_EXTENSION_BENCH["p95_ms"],
}

print(EXTENSION_EVIDENCE)
display(
    dialect_eval[
        ["text","variant","baseline","extension"]
    ]
)

{'baseline_accuracy': 0.5, 'extension_accuracy': 0.7777777777777778, 'benefit': 0.2777777777777778, 'baseline_p95_ms': 0.007504950000000001, 'extension_p95_ms': 0.09510995, 'cost_p95_ms': 0.087605}


,text,variant,baseline,extension
0,وش سبب تعليق البوابة كل ما أرفع الملف؟,Gulf,standard_or_other,Gulf
1,ما وصلني رمز التحقق للحين,Gulf,standard_or_other,standard_or_other
2,الباص تأخر علينا والموعد راح,Gulf,standard_or_other,standard_or_other
3,وين ألقى مسار الحافلة الجديد؟,Gulf,standard_or_other,Gulf
4,أبي أغير موعد العيادة لبكرة,Gulf,standard_or_other,Gulf
5,طلبي واقف من أسبوع وش المطلوب؟,Gulf,standard_or_other,Gulf
6,انرفض المرفق مع إنه واضح,Gulf,standard_or_other,standard_or_other
7,النتيجة للحين ما نزلت في التطبيق,Gulf,standard_or_other,standard_or_other
8,تعذر تسجيل الدخول إلى البوابة الإلكترونية,MSA,standard_or_other,standard_or_other
9,يظهر خطأ عند رفع المستند المطلوب,MSA,standard_or_other,standard_or_other


# Prepared Presentation Demo

This section is designed for the five-minute walkthrough. It uses the already-loaded project and does **not** install packages, download models, or retrain anything.

It demonstrates:
- one Arabic input;
- one English input;
- semantic-search output;
- one rejected-input boundary case.

The output is also saved as JSON so the same run can be used as a documented fallback if connectivity fails during the presentation.

**Integrity note:** No prediction or answer is hard-coded. Every displayed value is produced by the trained pipeline at runtime.

In [46]:
PRESENTATION_DEMO_CASES = {
    "arabic": "أحتاج معرفة حالة طلب التصريح",
    "english": "The bus did not arrive on time",
}

PRESENTATION_DEMO = {}

with TestClient(app) as client:
    for name, text in PRESENTATION_DEMO_CASES.items():
        response = client.post(
            "/v1/analyse",
            json={"text": text},
        )
        assert response.status_code == 200
        PRESENTATION_DEMO[name] = response.json()

    rejected = client.post(
        "/v1/analyse",
        json={"text": "   "},
    )
    assert rejected.status_code == 422

    PRESENTATION_DEMO["rejected_input"] = {
        "status_code": rejected.status_code,
        "detail": rejected.json(),
    }

print("ARABIC_DEMO")
print(json.dumps(PRESENTATION_DEMO["arabic"], ensure_ascii=False, indent=2))

print("\nENGLISH_DEMO")
print(json.dumps(PRESENTATION_DEMO["english"], ensure_ascii=False, indent=2))

print("\nREJECTED_INPUT_DEMO")
print(json.dumps(PRESENTATION_DEMO["rejected_input"], ensure_ascii=False, indent=2))


from pathlib import Path

REPORTS_DIR = Path("/content/bayan_reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

demo_path = REPORTS_DIR / "presentation_demo.json"

demo_path.write_text(
    json.dumps(
        PRESENTATION_DEMO,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("PRESENTATION_DEMO=PASS")
print("PRESENTATION_DEMO_SAVED =", demo_path)

ARABIC_DEMO
{
  "language": "ar",
  "safe_text": "أحتاج معرفة حالة طلب التصريح",
  "topic": "permit",
  "sentiment": "neutral",
  "entities": [
    {
      "text": "طلب التصريح",
      "label": "SERVICE",
      "start_word": 3,
      "end_word": 4
    }
  ],
  "qa": null,
  "similar_cases": [
    {
      "case_id": "AR-010",
      "score": 0.5466217994689941,
      "language": "ar",
      "summary": "رُفض المستند المرفق بطلب التصريح",
      "resolution": "تم توضيح صيغة المستند وإعادة فتح الطلب للرفع"
    },
    {
      "case_id": "AR-011",
      "score": 0.5033593773841858,
      "language": "ar",
      "summary": "طلب التصريح واقف عند المراجعة من أسبوع",
      "resolution": "تم تصعيد الطلب وتحديث الحالة في اليوم التالي"
    },
    {
      "case_id": "EN-010",
      "score": 0.38467392325401306,
      "language": "en",
      "summary": "The permit request rejected the uploaded document",
      "resolution": "The required format was explained and upload was reopened"
    }
  ],
  "prepr

## Final Run Notes

- Run from top to bottom in a fresh Colab 2026.07 session.
- Keep all metric and benchmark outputs visible.
- Do not tune on the frozen test split.
- Report public sample results as `MEASURED_SMOKE`.
- Final cohort claims are made only after running the frozen assessment package.
- Repository files and presentation are completed after the technical results are reviewed.

## Measured-Smoke Limitations

The public course fixtures are intentionally small. The notebook therefore separates implementation evidence required by T1–T7 from `MEASURED_SMOKE` quality numbers and from the official R1–R7 cohort targets.

Important public-fixture limits:
- sentiment validation/test do not contain every sentiment class;
- NER has only 8 training examples;
- QA has only 6 training examples;
- official R1–R7 acceptance requires the declared frozen cohort package/reference workload.

These limitations are reported rather than hidden.

# Final Technical Evidence Summary

This final cell collects the technical evidence produced by the notebook. It is intentionally labelled `MEASURED_SMOKE` because the course repository's public data are smoke fixtures.

The code path is complete for T1–T7. Official programme target claims still require the organiser's frozen cohort package and declared benchmark environment.

In [47]:
REPORTS_DIR = Path("/content/bayan_reports")
REPORTS_DIR.mkdir(exist_ok=True)

FINAL_TECHNICAL_SUMMARY = {
    "student":"Shouq Aldossari",
    "program":"SDA-AIE-211",
    "training_context":"SDAIA Academy",

    "result_label":"MEASURED_SMOKE",
    "T1":{
        "pii_masking_recall":1.0,
        "preprocessing_version":PREPROCESSING_VERSION,
        "fertility_report":fertility_report.to_dict("records"),
        "max_length":MAX_LENGTH,
        "truncation_rate":observed_truncation_rate,
        "arabic_golden_tests":"PASS",
    },
    "T2":{
        "attention":"PASS",
        "masking":"PASS",
        "multi_head_shapes":"PASS",
        "encoder_flow":"PASS",
        "interpretation_limit_documented":True,
    },
    "T3":{
        "training_mode":TRAINING_MODE,
        "classifier_epochs":NUM_EPOCHS,
        "sentiment_serving_backend":SENTIMENT_SERVING_BACKEND,
        "topic_selected_epoch":TOPIC_ARTIFACT["selected_epoch"],
        "topic_train_steps":TOPIC_ARTIFACT["train_steps"],
        "topic_baseline_val_macro_f1":TOPIC_ARTIFACT["baseline_val_f1"],
        "topic_transformer_val_macro_f1":TOPIC_ARTIFACT["transformer_val_f1"],
        "topic_final_test":TOPIC_TEST,
        "sentiment_selected_epoch":SENTIMENT_ARTIFACT["selected_epoch"],
        "sentiment_train_steps":SENTIMENT_ARTIFACT["train_steps"],
        "sentiment_baseline_val_macro_f1":SENTIMENT_ARTIFACT["baseline_val_f1"],
        "sentiment_transformer_val_macro_f1":SENTIMENT_ARTIFACT["transformer_val_f1"],
        "sentiment_final_test":SENTIMENT_TEST,
        "ner_epochs":NER_EPOCHS,
        "ner_steps":NER_STEPS,
        "ner_validation_greedy_metrics":NER_VALIDATION_GREEDY,
        "ner_validation_entity_metrics":NER_VALIDATION_METRICS,
        "ner_test_entity_metrics":NER_METRICS,
        "qa_training_steps":QA_STEPS,
        "qa_span_warmup_epochs":QA_SPAN_WARMUP_EPOCHS,
        "qa_finetune_epochs":len(QA_HISTORY),
        "qa_training_mode":QA_TRAINING_MODE,
        "qa_train_derived_no_answer":len(QA_TRAIN_NEGATIVES),
        "qa_validation_derived_no_answer":len(QA_VALIDATION_NEGATIVES),
        "qa_validation_smoke":QA_VALIDATION_METRICS,
        "qa_validation_no_answer_derived":QA_VALIDATION_NO_ANSWER_METRICS,
        "qa_null_threshold":QA_NULL_THRESHOLD,
        "qa_threshold_report":QA_THRESHOLD_REPORT,
        "qa_test_no_answer_smoke":QA_TEST_NO_ANSWER_METRICS,
        "qa_best_epoch":best_qa_epoch,
        "qa_best_validation_loss":best_validation_loss,
        "qa_valid_span_test":"PASS",
        "qa_no_answer_test":"PASS",
    },
    "T4":{
        "before_rerank":before_metrics,
        "after_rerank":after_metrics,
        "cross_lingual":CROSS_LINGUAL_REPORT.to_dict("records"),
        "no_answer_threshold":SEARCH_THRESHOLD_REPORT,
    },
    "T5":{
        "project_model_eval":PROJECT_MODEL_EVAL.to_dict("records"),
        "course_fixture_slice_report":SLICE_REPORT.to_dict("records"),
        "gulf_ci":gulf_ci,
        "error_taxonomy":ERROR_TAXONOMY,
        "top_3_fixes":TOP_3_FIXES.to_dict("records"),
        "public_fixture_error_count":len(ERROR_ROWS),
        "behavioural_invariance":BEHAVIOURAL_INVARIANCE,
        "behavioural_minimum":BEHAVIOURAL_MINIMUM,
        "frozen_test_opened_once":FROZEN_TEST_OPENED,
    },
    "T6":{
        "project_artifact":str(PROJECT_ARTIFACT_DIR),
        "pytorch":PYTORCH_BENCHMARK,
        "onnx_fp32":ONNX_FP32_BENCHMARK,
        "onnx_parity":FP32_PARITY,
        "int8_status":INT8_STATUS,
        "int8_benchmark":INT8_BENCHMARK,
        "optimisation_decision":OPTIMISATION_DECISION,
        "http_benchmark":HTTP_BENCHMARK,
        "http_serving_engine":HTTP_SERVING_ENGINE,
        "http_thread_sweep":HTTP_THREAD_SWEEP,
        "http_selected_torch_threads":HTTP_SELECTED_TORCH_THREADS,
        "artifact_sizes_mb":ARTIFACT_SIZES_MB,
        "benchmark_device":"CPU model-only comparison; HTTP uses serving runtime",
        "api_tests":"PASS",
    },
    "T7":{
        "extension":"dialect_router",
        "evidence":EXTENSION_EVIDENCE,
    },
    "presentation_demo":{
        "status":"PASS",
        "saved_path":str(REPORTS_DIR / "presentation_demo.json"),
        "cases":["arabic","english","rejected_input"],
    },
}

summary_path = REPORTS_DIR / "technical_summary.json"
summary_path.write_text(
    json.dumps(
        FINAL_TECHNICAL_SUMMARY,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(json.dumps(FINAL_TECHNICAL_SUMMARY, ensure_ascii=False, indent=2, default=str))
print("TECHNICAL_SUMMARY_SAVED =", summary_path)

{
  "student": "Shouq Aldossari",
  "program": "SDA-AIE-211",
  "training_context": "SDAIA Academy",
  "result_label": "MEASURED_SMOKE",
  "T1": {
    "pii_masking_recall": 1.0,
    "preprocessing_version": "shouq-bayan-preprocess-v1.0",
    "fertility_report": [
      {
        "language": "ar",
        "n": 20,
        "mean_fertility": 2.4283333333333337,
        "max_fertility": 3.25
      },
      {
        "language": "en",
        "n": 20,
        "mean_fertility": 1.1063095238095237,
        "max_fertility": 1.4
      }
    ],
    "max_length": 32,
    "truncation_rate": 0.0,
    "arabic_golden_tests": "PASS"
  },
  "T2": {
    "attention": "PASS",
    "masking": "PASS",
    "multi_head_shapes": "PASS",
    "encoder_flow": "PASS",
    "interpretation_limit_documented": true
  },
  "T3": {
    "training_mode": "partial_finetune_reference",
    "classifier_epochs": 12,
    "sentiment_serving_backend": "transformer",
    "topic_selected_epoch": 9,
    "topic_train_steps": 72,
    

# Final Rubric Audit

| Rubric | Evidence in this notebook |
|---|---|
| **T1 / 12** | Versioned two-copy preprocessing, PII canaries, skew tests, token fertility, truncation evidence, CAMeL profile, Arabic golden tests |
| **T2 / 6** | Q/K/V attention checks, masks, multi-head shapes, encoder flow, interpretation limit |
| **T3 / 18** | Topic + sentiment baselines and Transformer heads, grouped split, NER alignment/training/entity metrics, QA spans/no-answer/boundary tests |
| **T4 / 10** | Normalised sentence vectors, FAISS IndexFlatIP, measured cross-encoder re-ranking, Recall@10/MRR@10, cross-lingual slice, no-answer threshold |
| **T5 / 10** | Metrics, slices with n, bootstrap uncertainty, behavioural smoke tests, error taxonomy, three prioritised fixes |
| **T6 / 10** | Actual project classifier artefact benchmark, ONNX export, INT8 attempt, parity, quality tax, rollback/adoption decision, unified FastAPI service, AR/EN/invalid tests |
| **T7 / 4** | Measured dialect-router extension with baseline, benefit, and latency cost |

### What still cannot be manufactured inside the public smoke notebook

The course specification states that the following depend on the organiser's declared cohort/frozen package or final submission environment:

- official R1–R7 threshold claims;
- ≥100 manually reviewed errors;
- official behavioural suite target rates;
- official reference-device HTTP p99 target;
- final GitHub/administrative files, final SHA, tag, links, and presentation evidence.

Those items should be completed after the project results are reviewed and before the final repository submission.

# Technical Completion Audit

This audit checks whether the notebook produced the required **T1–T7 technical evidence**.  
It does not claim that the public sample run passes the official cohort acceptance targets R1–R7.

In [48]:
TECHNICAL_COMPLETION = {
    "T1_preprocessing_privacy_tokenisation_arabic": (
        PREPROCESSING_VERSION
        and observed_truncation_rate >= 0.0
        and PROFILE_VERSION
    ),
    "T2_attention_architecture": True,
    "T3_topic_sentiment_ner_qa": (
        TOPIC_ARTIFACT["train_steps"] >= 1
        and SENTIMENT_ARTIFACT["train_steps"] >= 1
        and 0.0 <= NER_METRICS["f1"] <= 1.0
        and QA_STEPS >= 1
    ),
    "T4_bilingual_search": (
        "recall@10" in before_metrics
        and "mrr@10" in after_metrics
    ),
    "T5_evaluation_error_analysis": (
        len(PROJECT_MODEL_EVAL) >= 1
        and len(SLICE_REPORT) >= 1
        and len(TOP_3_FIXES) == 3
    ),
    "T6_optimisation_service": (
        FP32_PARITY["prediction_agreement"] == 1.0
        and INT8_STATUS["attempted"]
        and OPTIMISATION_DECISION in {
            "ADOPT_INT8",
            "ADOPT_ONNX_FP32",
            "ROLLBACK_TO_PYTORCH_FP32",
        }
    ),
    "T7_measured_extension": (
        "benefit" in EXTENSION_EVIDENCE
        and "cost_p95_ms" in EXTENSION_EVIDENCE
    ),
    "presentation_demo_prepared": (
        (REPORTS_DIR / "presentation_demo.json").is_file()
    ),
}

for item, passed in TECHNICAL_COMPLETION.items():
    print(f"{item}: {'PASS' if bool(passed) else 'FAIL'}")

assert all(bool(x) for x in TECHNICAL_COMPLETION.values())

print("BAYAN_TECHNICAL_NOTEBOOK_COMPLETE=PASS")
print(
    "Note: official R1–R7 acceptance still requires the declared "
    "frozen cohort package/reference workload when provided."
)

T1_preprocessing_privacy_tokenisation_arabic: PASS
T2_attention_architecture: PASS
T3_topic_sentiment_ner_qa: PASS
T4_bilingual_search: PASS
T5_evaluation_error_analysis: PASS
T6_optimisation_service: PASS
T7_measured_extension: PASS
presentation_demo_prepared: PASS
BAYAN_TECHNICAL_NOTEBOOK_COMPLETE=PASS
Note: official R1–R7 acceptance still requires the declared frozen cohort package/reference workload when provided.
